# 🧬 MediLink RAG — Final Edition

**Biomedical RAG + HTAN image segmentation + Claude routing and cited answers**

This notebook is organized as a production-style runtime notebook. It removes duplicate implementations, automatic demos, repeated retrieval passes, and evaluation cells that executed unintentionally.

## Normal use

1. Set `PUBMED_EMAIL` and `ANTHROPIC_API_KEY` in the environment.
2. Start Weaviate and confirm the indexed collection exists.
3. Run the **Complete Startup** cell directly below.
4. Use the interactive control panel displayed at the end.

The startup cell executes only cells tagged as runtime stages. Installation, corpus construction, and manual evaluation do not run automatically.


### Evaluation
The final notebook also contains six optional evaluation tests. They are excluded from Complete Startup and run only when explicitly enabled.

In [ ]:

# =====================================================================
# RUN THE COMPLETE MEDILINK APPLICATION
# =====================================================================
# Run this cell after opening the notebook. It executes every startup
# cell below in the correct order and stops at the first failure.

import json
import os
import time
import traceback
from pathlib import Path
from IPython import get_ipython


def _find_medilink_notebook() -> Path:
    explicit = os.environ.get("MEDILINK_NOTEBOOK_PATH", "").strip()
    candidates = []

    if explicit:
        candidates.append(Path(explicit).expanduser())

    candidates.extend([
        Path.cwd() / "medilink_rag_FINAL_EDITION.ipynb",
        Path.cwd() / "medilink_rag_FINAL_v3.ipynb",
        Path.home() / "projects" / "medilink_rag_htan" / "medilink_rag_FINAL_EDITION.ipynb",
        Path.home() / "projects" / "medilink_rag_htan" / "medilink_rag_FINAL_v3.ipynb",
    ])

    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()

    for candidate in Path.cwd().glob("*.ipynb"):
        try:
            data = json.loads(candidate.read_text(encoding="utf-8"))
            if data.get("metadata", {}).get("medilink_project") is True:
                return candidate.resolve()
        except Exception:
            continue

    raise FileNotFoundError(
        "Could not locate the saved MediLink notebook. Save this notebook, "
        "or set MEDILINK_NOTEBOOK_PATH to its full path."
    )


def run_medilink_startup(show_code_preview: bool = False):
    notebook_path = _find_medilink_notebook()
    notebook = json.loads(notebook_path.read_text(encoding="utf-8"))

    startup_cells = []
    for index, cell in enumerate(notebook.get("cells", [])):
        order = cell.get("metadata", {}).get("medilink_startup_order")
        if order is None:
            continue
        if cell.get("cell_type") != "code":
            raise TypeError(
                f"Startup item {order} is not a code cell (notebook index {index})."
            )
        source = cell.get("source", "")
        source = "".join(source) if isinstance(source, list) else str(source)
        startup_cells.append((int(order), index, source))

    startup_cells.sort(key=lambda item: item[0])

    if not startup_cells:
        raise RuntimeError("No startup-tagged cells were found.")

    ipython = get_ipython()
    if ipython is None:
        raise RuntimeError("This runner must execute inside Jupyter/IPython.")

    print("=" * 84)
    print("🧬 MEDILINK FINAL STARTUP")
    print("=" * 84)
    print(f"Notebook : {notebook_path}")
    print(f"Stages   : {len(startup_cells)}")
    print()

    report = []
    total_start = time.perf_counter()

    for step, (order, notebook_index, source) in enumerate(startup_cells, 1):
        first_line = next(
            (
                line.strip("# ").strip()
                for line in source.splitlines()
                if line.strip() and not line.strip().startswith("# ===")
            ),
            f"Notebook cell {notebook_index + 1}",
        )

        print("-" * 84)
        print(f"▶ {step:02d}/{len(startup_cells):02d} | Stage {order:02d} | {first_line}")
        if show_code_preview:
            preview = "\n".join(source.splitlines()[:5])
            print(preview)

        started = time.perf_counter()
        result = ipython.run_cell(
            source,
            store_history=False,
            silent=False,
        )
        elapsed = time.perf_counter() - started

        success = getattr(result, "error_before_exec", None) is None and getattr(
            result, "error_in_exec", None
        ) is None

        report.append({
            "order": order,
            "notebook_index": notebook_index,
            "label": first_line,
            "success": success,
            "seconds": elapsed,
        })

        if not success:
            print(f"❌ Stage {order} failed after {elapsed:.2f}s")
            raise RuntimeError(
                f"MediLink startup stopped at stage {order}: {first_line}"
            )

        print(f"✅ Completed in {elapsed:.2f}s")

    total_elapsed = time.perf_counter() - total_start
    globals()["MEDILINK_STARTUP_REPORT"] = report
    globals()["MEDILINK_STARTUP_COMPLETE"] = True

    print()
    print("=" * 84)
    print(f"🚀 MediLink ready | {len(report)}/{len(report)} stages | {total_elapsed:.2f}s")
    print("=" * 84)

    return report


MEDILINK_STARTUP_REPORT = run_medilink_startup(show_code_preview=False)


## Runtime sequence

| Stage | Component | Purpose |
|---:|---|---|
| 1 | Configuration | Paths, models, thresholds, feature flags |
| 2 | Embedding model | Load S-PubMedBERT |
| 3 | Weaviate | Connect and verify indexed collection |
| 4 | Claude | Initialize Haiku/Sonnet wrappers and system prompts |
| 5 | Intent | Classify symptoms, treatment, mechanism, diagnosis, etc. |
| 6 | Hybrid search | Dense + BM25 + optional MMR |
| 7 | Query enhancement | Rewriting, HyDE, multi-query, decomposition |
| 8 | Reranking | MedCPT and optional lazy NLI |
| 9 | Advanced retrieval | One panel-controlled retrieval pipeline |
| 10 | OOS gate | Restrict the project scope |
| 11 | Answer safety | Citations, grounding, and disclaimers |
| 12 | Evaluation utilities | Metrics and benchmark definitions only |
| 13 | Triage | Personal-symptom clarification |
| 14 | Router | Route text and multimodal inputs |
| 15 | HTAN | Image segmentation and retrieval augmentation |
| 16 | Control panel | Interactive application UI |


## Stage 1 — Core Configuration
Defines all paths, models, thresholds, and runtime feature flags.


In [ ]:

# =====================================================================
# 01 — CORE CONFIGURATION
# =====================================================================
from pathlib import Path
import os
import json
import time
import re
import gc
import numpy as np
import requests
from tqdm.auto import tqdm
from Bio import Entrez
from sklearn.metrics.pairwise import cosine_similarity

import nltk

for _resource, _download_name in [
    ("tokenizers/punkt", "punkt"),
    ("tokenizers/punkt_tab", "punkt_tab"),
]:
    try:
        nltk.data.find(_resource)
    except LookupError:
        nltk.download(_download_name, quiet=True)


# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------
PROJECT_ROOT = Path.cwd().resolve()
DATA_ROOT = Path(os.environ.get("MEDILINK_DATA_ROOT", str(Path.home()))).expanduser().resolve()

PATH_DATASET     = str(DATA_ROOT / "pubmed_dataset.json")
PATH_WITH_PMCID  = str(DATA_ROOT / "pubmed_with_pmcid.json")
PATH_FULL_PAPERS = str(DATA_ROOT / "full_papers.json")
PATH_CLEANED     = str(DATA_ROOT / "cleaned_papers.json")
PATH_CHUNKS      = str(DATA_ROOT / "chunks_spubmedbert.json")
PATH_EMBEDDINGS  = str(DATA_ROOT / "embeddings_spubmedbert.npy")
PDF_DIR          = str(DATA_ROOT / "pdf_data")

# ---------------------------------------------------------------------
# User / service configuration
# ---------------------------------------------------------------------
EMAIL = os.environ.get("PUBMED_EMAIL", "your_email@example.com").strip()
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "").strip()
DATE_FROM = os.environ.get("MEDILINK_DATE_FROM", "2015")
DATE_TO = os.environ.get("MEDILINK_DATE_TO", "2025")

KEYWORDS = [
    "melanoma signs symptoms warning",
    "melanoma ABCDE early detection",
    "skin cancer warning signs mole",
    "melanoma diagnosis clinical features",
    "multiple myeloma symptoms bone pain fatigue",
    "multiple myeloma diagnosis signs",
    "skin cancer types symptoms treatment",
    "basal cell carcinoma signs symptoms",
    "squamous cell carcinoma symptoms",
    "melanoma prevention UV protection",
]

# ---------------------------------------------------------------------
# Models
# ---------------------------------------------------------------------
EMBEDDING_MODEL = "pritamdeka/S-PubMedBert-MS-MARCO"
EMBEDDING_DIM = 768

HAIKU_MODEL = "claude-haiku-4-5-20251001"
SONNET_MODEL = "claude-sonnet-4-6"

CROSS_ENCODER_MODEL = "ncbi/MedCPT-Cross-Encoder"
NLI_MODEL_NAME = "cross-encoder/nli-deberta-v3-small"

# ---------------------------------------------------------------------
# Retrieval
# ---------------------------------------------------------------------
INTENT_ALPHA = {
    "symptoms":   0.20,
    "treatment":  0.70,
    "mechanism":  0.75,
    "diagnosis":  0.45,
    "prognosis":  0.65,
    "medication": 0.50,
    "general":    0.60,
}

SEARCH_ALPHA = 0.60
SEARCH_K = 4
FETCH_K_MULT = 20
SEARCH_MMR = True
SEARCH_LAMBDA = 0.40

PATIENT_SOURCES = ["nhs", "cancer_gov", "medlineplus", "pdf"]

COMPRESSION_THRESHOLD = 0.50
SYMPTOM_BOOST = 0.15

OOS_THRESHOLD = 0.70
OOS_BORDERLINE_FACTOR = 0.70

USE_QUERY_REWRITER = True
USE_HYDE = False
USE_MULTI_QUERY = True
N_MULTI_QUERIES = 2
USE_DECOMPOSITION = False
USE_CROSS_ENCODER = True
USE_COMPRESSION = True
USE_NLI_FILTER = False
USE_HALLUC_CHECK = True
USE_AUTO_TRIAGE = True
USE_OOS_GATE = True
DISEASE_FILTER = None

# Panel uses these flags to make slider values override intent defaults.
PANEL_ALPHA_OVERRIDE_ACTIVE = False
PANEL_COMPRESSION_OVERRIDE_ACTIVE = False

# ---------------------------------------------------------------------
# Weaviate
# ---------------------------------------------------------------------
WEAVIATE_HOST = "localhost"
WEAVIATE_PORT = 8080
WEAVIATE_GRPC = 50051
COLLECTION_NAME = "PubMedChunksSPubMedBert"

# ---------------------------------------------------------------------
# Chunking metadata retained for reproducibility
# ---------------------------------------------------------------------
SENTENCE_WINDOW = 2
MERGE_THRESHOLD = 0.60
MAX_CHUNKS_PER_DOC = 30
MAX_CHUNKS_PDF = 100

SYMPTOM_VOCAB = {
    "presents with", "characterized by", "signs", "symptoms",
    "clinical manifestation", "warning sign", "abcde", "diagnosis",
    "patients notice", "visible", "appearance", "lesion", "mole",
    "itching", "bleeding", "color change", "border", "asymmetry",
    "pigmentation", "irregular", "ulceration", "nodule", "papule",
    "clinical feature", "physical examination", "dermoscopy",
    "bone pain", "fatigue", "weakness", "anemia", "infection",
    "night sweat", "weight loss", "back pain", "numbness",
}

# ---------------------------------------------------------------------
# Runtime state
# ---------------------------------------------------------------------
claude = None
client = None
collection = None
embed_model = None
cross_encoder = None
nli_model = None

conversation_history = []
MAX_HISTORY_TURNS = 10

Entrez.email = EMAIL

print("✅ Configuration loaded")
print(f"   Project root : {PROJECT_ROOT}")
print(f"   Data root    : {DATA_ROOT}")
print(f"   Embedding    : {EMBEDDING_MODEL}")
print(f"   Collection   : {COLLECTION_NAME}")
print(f"   API enabled  : {bool(ANTHROPIC_API_KEY)}")


## Stage 2 — Biomedical Embedding Model
Loads S-PubMedBERT and validates its output dimension.


In [ ]:

# =====================================================================
# 02 — LOAD BIOMEDICAL EMBEDDING MODEL
# =====================================================================
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"

if (
    globals().get("embed_model") is None
    or globals().get("_LOADED_EMBEDDING_MODEL") != EMBEDDING_MODEL
):
    print(f"⏳ Loading {EMBEDDING_MODEL} on {device.upper()}...")
    embed_model = SentenceTransformer(EMBEDDING_MODEL, device=device)
    _LOADED_EMBEDDING_MODEL = EMBEDDING_MODEL
else:
    print(f"✅ Reusing loaded embedding model: {EMBEDDING_MODEL}")

test_vector = np.asarray(
    embed_model.encode(
        ["MediLink biomedical embedding check"],
        normalize_embeddings=True,
        show_progress_bar=False,
    )
)

if test_vector.ndim != 2 or test_vector.shape[1] != EMBEDDING_DIM:
    raise RuntimeError(
        f"Embedding dimension mismatch: expected {EMBEDDING_DIM}, "
        f"received {test_vector.shape}"
    )

print(f"✅ Embedding model ready | device={device} | dimension={test_vector.shape[1]}")


## Stage 3 — Weaviate Connection
Connects to the existing vector collection without rebuilding or reinserting data.


In [ ]:

# =====================================================================
# 03 — CONNECT TO WEAVIATE
# =====================================================================
import weaviate
import weaviate.classes.query as wq

try:
    previous_client = globals().get("client")
    if previous_client is not None:
        previous_client.close()
except Exception:
    pass

client = weaviate.connect_to_local(
    host=WEAVIATE_HOST,
    port=WEAVIATE_PORT,
    grpc_port=WEAVIATE_GRPC,
)

try:
    meta = client.get_meta()
except Exception as error:
    raise RuntimeError(
        "Could not connect to Weaviate. Start the Weaviate service and rerun "
        "the startup cell."
    ) from error

if not client.collections.exists(COLLECTION_NAME):
    raise RuntimeError(
        f"Weaviate is connected, but collection '{COLLECTION_NAME}' does not exist. "
        "Build/import the knowledge base before running the application."
    )

collection = client.collections.get(COLLECTION_NAME)
collection_count = collection.aggregate.over_all(total_count=True).total_count

if collection_count <= 0:
    raise RuntimeError(
        f"Collection '{COLLECTION_NAME}' exists but contains no objects."
    )

print(f"✅ Connected to Weaviate v{meta.get('version', 'unknown')}")
print(f"✅ Collection ready: {COLLECTION_NAME} | objects={collection_count:,}")


## Stage 4 — Claude Client and System Prompts
Initializes Haiku/Sonnet wrappers. Local retrieval still works when the API is unavailable.


In [ ]:

# =====================================================================
# 04 — CLAUDE CLIENT AND SHARED SYSTEM PROMPTS
# =====================================================================
import anthropic

CITATION_SYSTEM = """
You are MediLink, a biomedical evidence-support assistant.

Use only the retrieved evidence supplied in the user prompt.

Rules:
1. Do not invent medical facts, statistics, treatments, diagnoses, or citations.
2. Cite every important medical claim with the exact supplied source identifier.
3. Use [PMID:12345678] when a PMID is available.
4. Clearly state when the retrieved evidence is insufficient.
5. Distinguish educational information from diagnosis or personalized advice.
6. Do not diagnose cancer from an image or from limited symptom information.
7. Recommend qualified clinical evaluation when appropriate.
""".strip()

DERMATOLOGY_SYSTEM = (
    "You are a careful dermatology diagnostic-support assistant. "
    "Do not diagnose from an image alone. Use retrieved evidence for medical claims."
)

if ANTHROPIC_API_KEY:
    claude = anthropic.Anthropic(
        api_key=ANTHROPIC_API_KEY,
        timeout=45.0,
        max_retries=1,
    )
    print("✅ Claude client configured")
else:
    claude = None
    print("⚠️ Claude client is disabled; local retrieval and rule-based fallbacks remain available.")


def _claude_text(response) -> str:
    blocks = getattr(response, "content", None) or []
    text_parts = [
        getattr(block, "text", "")
        for block in blocks
        if getattr(block, "text", "")
    ]
    return "\n".join(text_parts).strip()


def call_haiku(
    prompt: str,
    max_tokens: int = 200,
    system: str = None,
    history: list = None,
) -> str:
    if claude is None:
        raise RuntimeError("Claude API is not configured.")

    messages = []
    if history:
        messages.extend(history[-MAX_HISTORY_TURNS * 2 :])
    messages.append({"role": "user", "content": str(prompt)})

    kwargs = {
        "model": HAIKU_MODEL,
        "max_tokens": int(max_tokens),
        "messages": messages,
    }
    if system:
        kwargs["system"] = system

    return _claude_text(claude.messages.create(**kwargs))


def call_sonnet(
    prompt: str,
    max_tokens: int = 1500,
    system: str = None,
    history: list = None,
) -> str:
    if claude is None:
        raise RuntimeError("Claude API is not configured.")

    messages = []
    if history:
        messages.extend(history[-MAX_HISTORY_TURNS * 2 :])
    messages.append({"role": "user", "content": str(prompt)})

    kwargs = {
        "model": SONNET_MODEL,
        "max_tokens": int(max_tokens),
        "messages": messages,
    }
    if system:
        kwargs["system"] = system

    return _claude_text(claude.messages.create(**kwargs))


def update_history(role: str, content: str) -> None:
    conversation_history.append({"role": role, "content": str(content)})
    max_messages = MAX_HISTORY_TURNS * 2
    if len(conversation_history) > max_messages:
        del conversation_history[:-max_messages]


print("✅ Claude wrappers ready: call_haiku(), call_sonnet(), update_history()")


## Stage 5 — Intent Classification
Selects retrieval behavior for symptoms, treatment, mechanism, diagnosis, prognosis, medication, or general questions.


In [ ]:
# =====================================================================
# CELL 9 — INTENT CLASSIFIER
# Central utility used by all downstream cells
# =====================================================================

INTENT_LABELS = [
    "symptoms",    # signs, symptoms, manifestations, warning signs
    "treatment",   # therapy, drugs, management, clinical trials
    "mechanism",   # pathogenesis, molecular, signaling, biology
    "diagnosis",   # tests, criteria, detection, screening
    "prognosis",   # survival, outcomes, mortality, staging
    "medication",  # specific drug questions, interactions, dosing
    "general",     # other medical
]

_INTENT_CACHE = {}

def classify_intent(question: str) -> str:
    """Classify medical query intent using Haiku (cached)."""
    if question in _INTENT_CACHE:
        return _INTENT_CACHE[question]

    if claude:
        result = call_haiku(
            f"""Classify the primary information need of this medical question.
Choose ONE:
- symptoms: asking about signs, symptoms, clinical manifestations, warning signs, what it looks like
- treatment: asking about therapy, drugs, management, clinical trials
- mechanism: asking about pathogenesis, molecular biology, how disease works
- diagnosis: asking about tests, diagnostic criteria, detection, screening
- prognosis: asking about survival, outcomes, staging, prognosis
- medication: asking about a specific drug, dose, interaction, side effects
- general: other medical question

Question: {question}
Reply with ONE word only.""",
            max_tokens=15
        )
        intent = result.strip().lower()
        if intent not in INTENT_LABELS:
            intent = "general"
    else:
        # Regex fallback
        q = question.lower()
        if any(w in q for w in ["sign","symptom","present","manifest","warning","look like","abcde","detect early"]):
            intent = "symptoms"
        elif any(w in q for w in ["treat","therapy","drug","chemo","immunotherapy","surgery"]):
            intent = "treatment"
        elif any(w in q for w in ["mechanism","pathway","molecular","mutation","gene","protein"]):
            intent = "mechanism"
        elif any(w in q for w in ["diagnos","test","screen","biopsy","imaging","criteria"]):
            intent = "diagnosis"
        elif any(w in q for w in ["prognos","surviv","outcome","stage","mortality"]):
            intent = "prognosis"
        else:
            intent = "general"

    _INTENT_CACHE[question] = intent
    return intent

def get_alpha_for_intent(intent: str) -> float:
    """Return intent-specific hybrid search alpha."""
    return INTENT_ALPHA.get(intent, SEARCH_ALPHA)

print("✅ Intent classifier ready: classify_intent()")


## Stage 6 — Hybrid Search
Runs BM25+dense retrieval with metadata filters and optional MMR.


In [ ]:

# =====================================================================
# 06 — INTENT-ADAPTIVE HYBRID SEARCH
# =====================================================================
import weaviate.classes.query as wq


def _active_alpha(intent: str, explicit_alpha=None) -> float:
    if explicit_alpha is not None:
        return float(explicit_alpha)
    if globals().get("PANEL_ALPHA_OVERRIDE_ACTIVE", False):
        return float(SEARCH_ALPHA)
    return float(get_alpha_for_intent(intent))


def _combine_filters(*filters):
    active = [f for f in filters if f is not None]
    if not active:
        return None
    combined = active[0]
    for item in active[1:]:
        combined = combined & item
    return combined


def semantic_search(
    query: str,
    k: int = None,
    alpha: float = None,
    mmr: bool = None,
    lambda_mult: float = None,
    disease_filter=None,
    intent: str = None,
    symptom_only: bool = False,
) -> list:
    """Hybrid BM25+dense retrieval with optional metadata filtering and MMR."""
    if not str(query).strip():
        return []

    k = int(k or SEARCH_K)
    mmr = SEARCH_MMR if mmr is None else bool(mmr)
    lambda_mult = SEARCH_LAMBDA if lambda_mult is None else float(lambda_mult)
    intent = intent or classify_intent(query)
    alpha = _active_alpha(intent, alpha)

    query_vec = np.asarray(
        embed_model.encode(
            query,
            normalize_embeddings=True,
            show_progress_bar=False,
        )
    )
    coll = client.collections.get(COLLECTION_NAME)
    fetch_k = max(k, k * FETCH_K_MULT) if mmr else k

    disease_clause = (
        wq.Filter.by_property("keyword").equal(disease_filter)
        if disease_filter
        else None
    )
    symptom_clause = (
        wq.Filter.by_property("chunk_type").equal("symptom")
        if symptom_only
        else None
    )
    filters = _combine_filters(disease_clause, symptom_clause)

    kwargs = {
        "query": query,
        "vector": query_vec.tolist(),
        "alpha": alpha,
        "limit": fetch_k,
    }
    if filters is not None:
        kwargs["filters"] = filters

    result = coll.query.hybrid(**kwargs)

    candidates = []
    for obj in result.objects:
        props = obj.properties or {}
        text = props.get("text", "")
        if not text:
            continue
        candidates.append({
            "text": text,
            "pmid": props.get("pmid", ""),
            "pmcid": props.get("pmcid", ""),
            "keyword": props.get("keyword", ""),
            "title": props.get("title", ""),
            "source": props.get("source", ""),
            "chunk_type": props.get("chunk_type", "general"),
        })

    if not mmr or len(candidates) <= k:
        return candidates[:k]

    cand_embs = np.asarray(
        embed_model.encode(
            [item["text"] for item in candidates],
            device="cpu",
            batch_size=32,
            normalize_embeddings=True,
            show_progress_bar=False,
        )
    )
    q_emb = query_vec.reshape(1, -1)

    selected = []
    remaining = list(range(len(candidates)))

    while remaining and len(selected) < k:
        if not selected:
            similarities = cosine_similarity(q_emb, cand_embs)[0]
            best = max(remaining, key=lambda idx: similarities[idx])
        else:
            selected_embs = cand_embs[selected]
            scored = []
            for idx in remaining:
                relevance = float(
                    cosine_similarity(
                        q_emb,
                        cand_embs[idx].reshape(1, -1),
                    )[0][0]
                )
                redundancy = float(
                    cosine_similarity(
                        cand_embs[idx].reshape(1, -1),
                        selected_embs,
                    ).max()
                )
                scored.append((
                    lambda_mult * relevance
                    - (1.0 - lambda_mult) * redundancy,
                    idx,
                ))
            best = max(scored, key=lambda pair: pair[0])[1]

        selected.append(best)
        remaining.remove(best)

    return [candidates[idx] for idx in selected]


def build_disease_filter(disease: str = None):
    """Return a Weaviate keyword filter for external callers."""
    if not disease:
        return None
    return wq.Filter.by_property("keyword").equal(disease)


print("✅ Hybrid search ready: semantic_search()")


## Stage 7 — Query Rewriting
Converts informal questions into biomedical retrieval language.


In [ ]:
import re

# =====================================================================
# CELL 14 — LLM QUERY REWRITER (INTENT-PRESERVING)
# =====================================================================
# Critical rule: NEVER strip intent words (signs, symptoms, warning)
# Expand them instead to match PubMed vocabulary

INTENT_EXPANSIONS = {
    "symptoms": "signs symptoms clinical manifestations clinical presentation patients present with",
    "treatment": "treatment therapy management intervention clinical trial",
    "mechanism": "mechanism pathogenesis molecular biology signaling pathway",
    "diagnosis": "diagnosis diagnostic criteria detection screening examination",
    "prognosis": "prognosis survival outcomes mortality staging",
}

REWRITER_SYSTEM = """You are a biomedical PubMed search query optimizer.

CRITICAL RULES — follow exactly:
1. NEVER remove intent words: signs, symptoms, warning, diagnosis, detect, present, manifest
2. EXPAND intent words: "signs" → "signs symptoms clinical manifestations clinical presentation"
3. ADD medical synonyms: use MeSH terminology alongside lay terms
4. EXPAND abbreviations: MM → multiple myeloma, BCC → basal cell carcinoma
5. Return a concise query (under 25 words)
6. Return ONLY the query, no explanation

Examples:
- "What are the signs of melanoma?" → "melanoma signs symptoms clinical manifestations warning ABCDE criteria presentation"
- "How does bortezomib work in MM?" → "bortezomib mechanism proteasome inhibitor multiple myeloma"
- "side effects of chemo for BCC?" → "basal cell carcinoma chemotherapy adverse events side effects toxicity"
"""

BIOMEDICAL_ABBREVIATIONS = {
    r"\bMM\b":"multiple myeloma", r"\bBCC\b":"basal cell carcinoma",
    r"\bSCC\b":"squamous cell carcinoma", r"\bASCT\b":"autologous stem cell transplant",
    r"\bCAR-?T\b":"CAR-T cell therapy", r"\bPD-?1\b":"PD-1 immune checkpoint",
    r"\bCTLA-?4\b":"CTLA-4 immune checkpoint", r"\bBCMA\b":"B-cell maturation antigen",
    r"\bSLNB\b":"sentinel lymph node biopsy", r"\bUV\b":"ultraviolet radiation",
    r"\birAEs?\b":"immune-related adverse events", r"\bVRd\b":"bortezomib lenalidomide dexamethasone",
    r"\bMohs\b":"Mohs micrographic surgery", r"\bPFS\b":"progression-free survival",
    r"\bOS\b":"overall survival", r"\bNMSC\b":"non-melanoma skin cancer",
    r"\bAK\b":"actinic keratosis", r"\bMCC\b":"Merkel cell carcinoma",
}
FILLER_PATTERNS = [
    r"^(?:can you |please |tell me |i want to know |could you |i(?:'d| would) like to know )+(about |regarding |on )?",
    r"(?:give me|provide me with|show me)\s+",
    r"\s+please$", r"\s+thanks?\.?$",
]

def _regex_rewrite(question: str, intent: str = None) -> str:
    q = question
    for pat in FILLER_PATTERNS:
        q = re.sub(pat, "", q, flags=re.IGNORECASE).strip()
    for pat, rep in BIOMEDICAL_ABBREVIATIONS.items():
        q = re.sub(pat, rep, q)
    # Expand intent terms
    if intent and intent in INTENT_EXPANSIONS:
        expansion = INTENT_EXPANSIONS[intent]
        q = q + " " + expansion
    q = re.sub(r"\s{2,}", " ", q).strip()
    if q and q[-1] not in ".?!":
        q += "?"
    return q

def rewrite_query(question: str, verbose: bool = False,
                  intent: str = None) -> str:
    """
    Intent-preserving query rewriter.
    Uses Haiku with strict rules to EXPAND intent vocabulary.
    Falls back to regex if API unavailable.
    """
    if not USE_QUERY_REWRITER:
        return question

    if intent is None:
        intent = classify_intent(question)
    # FIXED: skip rewriting symptom queries — source filter handles intent
    if intent == "symptoms":
        if verbose: print(f"  [Rewriter:symptoms] skipped (source filter active)")
        return question

    original = question

    if claude:
        q = call_haiku(
            f"Intent detected: {intent}\n\nRewrite for PubMed: {question}",
            max_tokens=60,
            system=REWRITER_SYSTEM
        )
        # Validate — must contain original disease terms
        q = re.sub(r"^(rewritten:?|query:?|result:?)\s*", "", q, flags=re.IGNORECASE).strip()
        if len(q) < 5 or len(q) > 200:
            q = _regex_rewrite(question, intent)
    else:
        q = _regex_rewrite(question, intent)

    if verbose and q != original:
        print(f"  [Rewriter:{intent}] '{original}'")
        print(f"                    → '{q}'")
    return q

print("✅ Query rewriter ready: rewrite_query()")


## Stage 8 — HyDE, Multi-Query, and Decomposition
Defines optional query-expansion techniques controlled by the panel.


In [ ]:
# =====================================================================
# CELL 15 — INTENT-CONDITIONED HyDE + MULTI-QUERY + DECOMPOSITION
# =====================================================================

HYDE_PROMPTS = {
    "symptoms": """Write a 3-sentence excerpt from a clinical dermatology or oncology review paper
describing the specific signs, symptoms, and clinical manifestations of {disease}.
Use phrases like: 'presents with', 'characterized by', 'clinical features include',
'patients may notice', 'physical examination reveals', 'ABCDE criteria'.
Focus ONLY on what is visible/felt by patients and observed by clinicians.""",

    "treatment": """Write a 3-sentence excerpt from a clinical trial abstract about
treatment of {disease}. Focus on: drug names, mechanisms, response rates, survival outcomes.""",

    "mechanism": """Write a 3-sentence excerpt from a molecular biology paper about
pathogenesis of {disease}. Focus on: gene mutations, signaling pathways, cellular mechanisms.""",

    "diagnosis": """Write a 3-sentence excerpt from a clinical guidelines paper about
diagnosis of {disease}. Focus on: diagnostic criteria, tests, imaging, biopsy, staging.""",

    "prognosis": """Write a 3-sentence excerpt from a survival analysis paper about
prognosis of {disease}. Focus on: survival rates, prognostic factors, staging, outcomes.""",

    "general": """Write a 3-sentence excerpt from a PubMed review paper that answers: {question}"""
}

MULTI_QUERY_SYSTEM = """Generate {n} biomedical PubMed search queries for this question.
REQUIRED: Each query must cover a structurally different angle.

For SYMPTOM questions, use these angles:
1. Patient perspective: what patients experience/notice
2. Clinical perspective: what physicians observe/diagnose
3. Technical/MeSH: medical terminology + MeSH headings
4. Differential: distinguishing features from similar conditions

For TREATMENT questions, use:
1. Drug mechanism of action
2. Clinical trial outcomes
3. Combination regimens
4. Resistance and alternatives

Return ONLY queries, one per line, no numbering or bullets."""


def _extract_disease(question: str) -> str:
    """Extract disease name from question."""
    diseases = ["melanoma","skin cancer","multiple myeloma","basal cell carcinoma",
                "squamous cell carcinoma","myeloma","BCC","SCC"]
    q = question.lower()
    for d in diseases:
        if d.lower() in q:
            return d
    return "cancer"  # fallback


def generate_hypothetical_doc(question: str, intent: str = None) -> str:
    """Intent-conditioned HyDE — generates relevant fake abstract."""
    if intent is None:
        intent = classify_intent(question)

    disease = _extract_disease(question)

    if claude:
        template = HYDE_PROMPTS.get(intent, HYDE_PROMPTS["general"])
        prompt   = template.format(disease=disease, question=question)
        hypo = call_haiku(prompt, max_tokens=150)
        return hypo if len(hypo) > 30 else question
    
    # Fallback template
    topic = re.sub(r"\?$","",question.lower()).strip()
    return (f"Clinical studies have investigated {topic}. "
            f"Evidence demonstrates specific {intent}-related findings for {disease}. "
            f"Treatment approaches and clinical outcomes are well-characterized.")


def hyde_query_vector(question: str, intent: str = None):
    """Embed intent-conditioned hypothetical document."""
    hypo = generate_hypothetical_doc(question, intent)
    return embed_model.encode(hypo)


def generate_multi_queries(question: str, n: int = N_MULTI_QUERIES,
                            intent: str = None) -> list:
    """Generate structurally diverse queries using Haiku."""
    if intent is None:
        intent = classify_intent(question)

    if claude:
        response = call_haiku(
            f"Intent: {intent}\nOriginal: {question}\n\nGenerate {n} queries:",
            max_tokens=200,
            system=MULTI_QUERY_SYSTEM.format(n=n)
        )
        queries = [q.strip() for q in response.strip().split("\n") if len(q.strip()) > 10]
        if queries:
            return [question] + queries[:n-1]

    # Fallback templates by intent
    q = question.strip().rstrip("?")
    if intent == "symptoms":
        templates = [
            question,
            f"clinical manifestations and presentation of {_extract_disease(question)}",
            f"warning signs {_extract_disease(question)} patients experience",
            f"{_extract_disease(question)} ABCDE criteria clinical features diagnosis",
        ]
    elif intent == "treatment":
        templates = [
            question,
            f"mechanism of action {q}",
            f"clinical trial outcomes {q}",
            f"combination therapy resistance {_extract_disease(question)}",
        ]
    else:
        templates = [
            question,
            f"What is the mechanism of {q}?",
            f"Clinical outcomes related to {q}?",
            f"Recent advances in {q}?",
        ]
    return templates[:max(n, 1)]


def decompose_query(question: str) -> list:
    """Decompose compound questions using Haiku."""
    if claude:
        response = call_haiku(
            f"""If this question contains multiple distinct medical sub-questions, split them.
If single question, return as-is.
Return questions one per line.

Question: {question}""",
            max_tokens=150
        )
        parts = [p.strip().rstrip("?").strip() + "?"
                 for p in response.strip().split("\n") if len(p.strip()) > 15]
        if len(parts) > 1:
            return parts

    # Regex fallback
    for pat in [r"(?i)\s+and\s+how\s+", r"(?i)\s+and\s+what\s+", r"(?i)\s+as\s+well\s+as\s+"]:
        parts = re.split(pat, question)
        if len(parts) > 1:
            return [p.strip().rstrip("?") + "?" for p in parts if len(p.strip()) > 15]
    return [question]

print("✅ Query expansion ready: HyDE, multi-query, decomposition")


## Stage 9 — Biomedical Reranking and Optional NLI
Models are loaded lazily only when the selected panel options require them.


In [ ]:

# =====================================================================
# 09 — BIOMEDICAL RERANKER AND OPTIONAL NLI FILTER
# =====================================================================
import torch
from sentence_transformers import CrossEncoder

_LOADED_CROSS_ENCODER_MODEL = None
_LOADED_NLI_MODEL = None
NLI_AVAILABLE = None


def ensure_cross_encoder(model_name: str = None):
    """Load or reuse the selected cross-encoder."""
    global cross_encoder, CROSS_ENCODER_MODEL, _LOADED_CROSS_ENCODER_MODEL

    model_name = model_name or CROSS_ENCODER_MODEL
    if cross_encoder is not None and _LOADED_CROSS_ENCODER_MODEL == model_name:
        return cross_encoder

    device_name = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"⏳ Loading cross-encoder: {model_name} on {device_name.upper()}")

    try:
        cross_encoder = CrossEncoder(
            model_name,
            device=device_name,
            max_length=512,
        )
        CROSS_ENCODER_MODEL = model_name
        _LOADED_CROSS_ENCODER_MODEL = model_name
        return cross_encoder
    except Exception as error:
        fallback = "cross-encoder/ms-marco-MiniLM-L-12-v2"
        if model_name == fallback:
            raise RuntimeError(
                f"Could not load cross-encoder '{model_name}'."
            ) from error

        print(f"⚠️ Could not load {model_name}; using {fallback}")
        cross_encoder = CrossEncoder(
            fallback,
            device=device_name,
            max_length=512,
        )
        CROSS_ENCODER_MODEL = fallback
        _LOADED_CROSS_ENCODER_MODEL = fallback
        return cross_encoder


def cross_encoder_rerank(
    query: str,
    docs: list,
    top_k: int = None,
) -> list:
    """Jointly score query-document pairs and return descending relevance."""
    if not docs:
        return []

    model = ensure_cross_encoder(CROSS_ENCODER_MODEL)
    pairs = [
        (str(query), str(doc.get("text", ""))[:1800])
        for doc in docs
    ]
    scores = np.asarray(
        model.predict(
            pairs,
            show_progress_bar=False,
        )
    ).reshape(-1)

    ranked = []
    for document, score in zip(docs, scores):
        item = dict(document)
        item["cross_encoder_score"] = float(score)
        ranked.append(item)

    ranked.sort(
        key=lambda item: item.get("cross_encoder_score", float("-inf")),
        reverse=True,
    )
    return ranked[:top_k] if top_k else ranked


def ensure_nli_model():
    """Lazily load the NLI model only when the panel enables NLI."""
    global nli_model, NLI_AVAILABLE, _LOADED_NLI_MODEL

    if nli_model is not None and _LOADED_NLI_MODEL == NLI_MODEL_NAME:
        NLI_AVAILABLE = True
        return nli_model

    device_name = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"⏳ Loading optional NLI model: {NLI_MODEL_NAME}")

    try:
        nli_model = CrossEncoder(
            NLI_MODEL_NAME,
            device=device_name,
            max_length=512,
        )
        _LOADED_NLI_MODEL = NLI_MODEL_NAME
        NLI_AVAILABLE = True
        return nli_model
    except Exception as error:
        NLI_AVAILABLE = False
        raise RuntimeError(
            f"Could not load NLI model '{NLI_MODEL_NAME}'."
        ) from error


def _nli_entailment_index(model) -> int:
    id2label = getattr(
        getattr(getattr(model, "model", None), "config", None),
        "id2label",
        {},
    ) or {}

    for index, label in id2label.items():
        if "entail" in str(label).lower():
            return int(index)

    # Common NLI ordering fallback. The model configuration is preferred.
    return 1


def nli_citation_filter(
    query: str,
    docs: list,
    min_nli_score: float = 0.40,
) -> list:
    """Keep passages whose premise entails the query/hypothesis."""
    if not docs:
        return []

    model = ensure_nli_model()
    pairs = [
        (str(doc.get("text", ""))[:1800], str(query))
        for doc in docs
    ]

    raw_scores = np.asarray(
        model.predict(
            pairs,
            apply_softmax=True,
            show_progress_bar=False,
        )
    )

    if raw_scores.ndim == 1:
        raw_scores = raw_scores.reshape(-1, 1)

    entailment_index = min(
        _nli_entailment_index(model),
        raw_scores.shape[1] - 1,
    )

    filtered = []
    for document, row in zip(docs, raw_scores):
        entailment_score = float(row[entailment_index])
        item = dict(document)
        item["nli_label"] = "ENTAILMENT"
        item["nli_score"] = entailment_score
        if entailment_score >= float(min_nli_score):
            filtered.append(item)

    if not filtered:
        # Never return an empty evidence set solely because of NLI.
        return sorted(
            docs,
            key=lambda item: item.get("cross_encoder_score", 0.0),
            reverse=True,
        )[:1]

    filtered.sort(
        key=lambda item: item.get("nli_score", 0.0),
        reverse=True,
    )
    return filtered


print("✅ Reranking utilities ready; models load lazily when selected.")


## Stage 10 — Advanced Retrieval Pipeline
Runs one retrieval pass using the exact panel choices.


In [ ]:

# =====================================================================
# 10 — ADVANCED INTENT-AWARE RETRIEVAL PIPELINE
# =====================================================================
from nltk.tokenize import sent_tokenize

CE_SOFT_CUTOFF = 0.01
MIN_FINAL_DOCS = 1


def compress_context(
    query: str,
    docs: list,
    threshold: float = None,
    intent: str = None,
) -> list:
    """Keep only query-relevant sentences while preserving source metadata."""
    if not docs:
        return []

    intent = intent or classify_intent(query)
    threshold = (
        COMPRESSION_THRESHOLD
        if threshold is None
        else float(threshold)
    )

    q_emb = np.asarray(
        embed_model.encode(
            [query],
            normalize_embeddings=True,
            show_progress_bar=False,
        )
    )

    compressed = []
    for document in docs:
        text = str(document.get("text", "")).strip()
        if not text:
            continue

        try:
            sentences = [
                sentence.strip()
                for sentence in sent_tokenize(text)
                if sentence.strip()
            ]
        except LookupError:
            sentences = [
                sentence.strip()
                for sentence in re.split(r"(?<=[.!?])\s+", text)
                if sentence.strip()
            ]

        if not sentences:
            compressed.append(dict(document))
            continue

        sentence_embeddings = np.asarray(
            embed_model.encode(
                sentences,
                normalize_embeddings=True,
                show_progress_bar=False,
            )
        )
        similarities = cosine_similarity(q_emb, sentence_embeddings)[0]

        kept = []
        for sentence, similarity in zip(sentences, similarities):
            boost = 0.0
            if intent == "symptoms":
                lower_sentence = sentence.lower()
                if any(term in lower_sentence for term in SYMPTOM_VOCAB):
                    boost = SYMPTOM_BOOST

            if float(similarity) + boost >= threshold:
                kept.append(sentence)

        if not kept:
            kept = [sentences[int(np.argmax(similarities))]]

        item = dict(document)
        item["original_text"] = text
        item["text"] = " ".join(kept)
        item["compression_ratio"] = round(
            len(kept) / max(len(sentences), 1),
            3,
        )
        compressed.append(item)

    return compressed


def _hybrid_search_raw(
    query: str,
    query_vector,
    k: int,
    alpha: float,
    disease_filter=None,
    source_filter=None,
) -> list:
    coll = client.collections.get(COLLECTION_NAME)
    fetch_k = max(k, k * FETCH_K_MULT)

    filters = None
    if disease_filter:
        filters = wq.Filter.by_property("keyword").equal(disease_filter)

    if source_filter:
        source_clause = wq.Filter.by_property("source").contains_any(
            list(source_filter)
        )
        filters = (
            source_clause
            if filters is None
            else filters & source_clause
        )

    kwargs = {
        "query": query,
        "vector": np.asarray(query_vector).tolist(),
        "alpha": float(alpha),
        "limit": int(fetch_k),
    }
    if filters is not None:
        kwargs["filters"] = filters

    result = coll.query.hybrid(**kwargs)

    documents = []
    for obj in result.objects:
        props = obj.properties or {}
        text = props.get("text", "")
        if not text:
            continue
        documents.append({
            "text": text,
            "pmid": props.get("pmid", ""),
            "pmcid": props.get("pmcid", ""),
            "keyword": props.get("keyword", ""),
            "title": props.get("title", ""),
            "source": props.get("source", ""),
            "chunk_type": props.get("chunk_type", "general"),
        })
    return documents


def _mmr_rerank(
    query_vector,
    candidates: list,
    k: int,
    lambda_mult: float = None,
) -> list:
    if not candidates:
        return []

    if len(candidates) <= k:
        return candidates

    lambda_mult = (
        SEARCH_LAMBDA
        if lambda_mult is None
        else float(lambda_mult)
    )

    candidate_embeddings = np.asarray(
        embed_model.encode(
            [item.get("text", "") for item in candidates],
            device="cpu",
            batch_size=32,
            normalize_embeddings=True,
            show_progress_bar=False,
        )
    )
    query_embedding = np.asarray(query_vector).reshape(1, -1)

    selected = []
    remaining = list(range(len(candidates)))

    while remaining and len(selected) < k:
        if not selected:
            similarities = cosine_similarity(
                query_embedding,
                candidate_embeddings,
            )[0]
            best = max(remaining, key=lambda idx: similarities[idx])
        else:
            selected_embeddings = candidate_embeddings[selected]
            scored = []
            for idx in remaining:
                relevance = float(
                    cosine_similarity(
                        query_embedding,
                        candidate_embeddings[idx].reshape(1, -1),
                    )[0][0]
                )
                redundancy = float(
                    cosine_similarity(
                        candidate_embeddings[idx].reshape(1, -1),
                        selected_embeddings,
                    ).max()
                )
                score = (
                    lambda_mult * relevance
                    - (1.0 - lambda_mult) * redundancy
                )
                scored.append((score, idx))
            best = max(scored, key=lambda item: item[0])[1]

        selected.append(best)
        remaining.remove(best)

    return [candidates[idx] for idx in selected]


def _select_final_docs_from_reranked(
    reranked: list,
    k: int,
) -> list:
    if not reranked:
        return []

    scored = [
        item
        for item in reranked
        if "cross_encoder_score" in item
    ]

    if not scored:
        return reranked[:k]

    strong = [
        item
        for item in reranked
        if float(item.get("cross_encoder_score", 0.0))
        >= CE_SOFT_CUTOFF
    ]

    return strong[:k] if strong else reranked[:1]


def advanced_retrieve(
    question,
    k=None,
    use_hyde=None,
    use_multi_q=None,
    use_cross_enc=None,
    use_compress=None,
    use_decomp=None,
    use_nli_filter=None,
    disease=None,
    verbose=False,
    intent=None,
    alpha=None,
) -> list:
    """Complete panel-controlled retrieval pipeline."""
    question = str(question).strip()
    if not question:
        return []

    k = int(k or SEARCH_K)
    use_hyde = USE_HYDE if use_hyde is None else bool(use_hyde)
    use_multi_q = (
        USE_MULTI_QUERY
        if use_multi_q is None
        else bool(use_multi_q)
    )
    use_cross_enc = (
        USE_CROSS_ENCODER
        if use_cross_enc is None
        else bool(use_cross_enc)
    )
    use_compress = (
        USE_COMPRESSION
        if use_compress is None
        else bool(use_compress)
    )
    use_decomp = (
        USE_DECOMPOSITION
        if use_decomp is None
        else bool(use_decomp)
    )
    use_nli_filter = (
        USE_NLI_FILTER
        if use_nli_filter is None
        else bool(use_nli_filter)
    )
    disease = DISEASE_FILTER if disease is None else disease
    intent = intent or classify_intent(question)

    active_alpha = _active_alpha(intent, alpha)

    if verbose:
        print(
            f"[Intent] {intent} | alpha={active_alpha:.2f} | "
            f"k={k} | disease={disease or 'all'}"
        )

    rewritten_question = question
    if USE_QUERY_REWRITER and "rewrite_query" in globals():
        rewritten_question = rewrite_query(
            question,
            intent=intent,
            verbose=verbose,
        )

    sub_questions = (
        decompose_query(rewritten_question)
        if use_decomp
        else [rewritten_question]
    )

    all_candidates = {}

    for sub_question in sub_questions:
        base_vector = (
            hyde_query_vector(sub_question, intent=intent)
            if use_hyde
            else embed_model.encode(
                sub_question,
                normalize_embeddings=True,
                show_progress_bar=False,
            )
        )

        query_variants = (
            generate_multi_queries(
                sub_question,
                n=N_MULTI_QUERIES,
                intent=intent,
            )
            if use_multi_q
            else [sub_question]
        )
        query_variants = list(
            dict.fromkeys([sub_question] + list(query_variants))
        )

        source_filter = PATIENT_SOURCES if intent == "symptoms" else None

        for retrieval_query in query_variants:
            results = _hybrid_search_raw(
                retrieval_query,
                base_vector,
                k=k,
                alpha=active_alpha,
                disease_filter=disease,
                source_filter=source_filter,
            )

            # If patient-only filtering returns nothing, fall back to the
            # same disease scope without source filtering.
            if not results and source_filter:
                results = _hybrid_search_raw(
                    retrieval_query,
                    base_vector,
                    k=k,
                    alpha=active_alpha,
                    disease_filter=disease,
                    source_filter=None,
                )

            for document in results:
                key = (
                    document.get("source", ""),
                    document.get("pmid", ""),
                    document.get("title", "")[:100],
                    document.get("text", "")[:160],
                )
                if key not in all_candidates:
                    item = dict(document)
                    item["_retrieval_count"] = 1
                    all_candidates[key] = item
                else:
                    all_candidates[key]["_retrieval_count"] += 1

    candidates = list(all_candidates.values())
    if not candidates:
        return []

    final_query_vector = (
        hyde_query_vector(rewritten_question, intent=intent)
        if use_hyde
        else embed_model.encode(
            rewritten_question,
            normalize_embeddings=True,
            show_progress_bar=False,
        )
    )

    mmr_keep = min(
        max(k * 4, k),
        len(candidates),
    )
    selected = _mmr_rerank(
        final_query_vector,
        candidates,
        k=mmr_keep,
        lambda_mult=SEARCH_LAMBDA,
    )

    if use_cross_enc:
        reranked = cross_encoder_rerank(
            rewritten_question,
            selected,
            top_k=max(k, min(len(selected), k * 2)),
        )
        final_docs = _select_final_docs_from_reranked(
            reranked,
            k,
        )
    else:
        final_docs = selected[:k]

    if use_nli_filter and final_docs:
        final_docs = nli_citation_filter(
            rewritten_question,
            final_docs,
        )[:k]

    if use_compress and final_docs:
        final_docs = compress_context(
            rewritten_question,
            final_docs,
            threshold=COMPRESSION_THRESHOLD,
            intent=intent,
        )

    if verbose:
        print(
            f"[Retrieve] candidates={len(candidates)} | "
            f"final={len(final_docs)} | "
            f"HyDE={use_hyde} | multi={use_multi_q} | "
            f"cross={use_cross_enc} | compression={use_compress} | "
            f"NLI={use_nli_filter}"
        )

    return final_docs


def build_advanced_rag_prompt(
    question: str,
    k: int = None,
    intent: str = None,
    **kwargs,
):
    """Retrieve evidence once and build a citation-grounded prompt."""
    intent = intent or classify_intent(question)
    docs = advanced_retrieve(
        question,
        k=k or SEARCH_K,
        intent=intent,
        **kwargs,
    )

    lines = [
        "Answer using only the retrieved evidence below.",
        "Cite every important medical claim using the exact source ID.",
        f"Intent: {intent}",
        "",
        "RETRIEVED EXCERPTS:",
    ]

    if not docs:
        lines.append("No relevant excerpts were retrieved.")
    else:
        for index, document in enumerate(docs, 1):
            source_id = (
                document.get("pmid")
                or document.get("source")
                or f"doc{index}"
            )
            lines.append(
                f"\n[{index}] PMID:{source_id}\n"
                f"Title: {document.get('title', '')}\n"
                f"Source: {document.get('source', '')}\n"
                f"Text: {document.get('text', '')}"
            )

    lines.extend([
        "",
        f"QUESTION: {question}",
        "",
        "ANSWER:",
    ])

    return "\n".join(lines), docs


print("✅ Advanced retrieval ready: advanced_retrieve(), build_advanced_rag_prompt()")


## Stage 11 — Out-of-Scope Detection
Combines retrieval evidence with Haiku or a rule-based fallback.


In [ ]:

# =====================================================================
# 11 — OUT-OF-SCOPE DETECTION
# =====================================================================
import json
import re

MEDILINK_SCOPE = """
MediLink answers questions about melanoma, skin cancer (including basal
cell carcinoma and squamous cell carcinoma), multiple myeloma, and their
symptoms, diagnosis, causes, risk factors, staging, prognosis, treatments,
side effects, biomarkers, mutations, immunotherapy, CAR-T, and targeted therapy.
""".strip()

OOS_CACHE = {}

OUT_OF_SCOPE_QUERIES = [
    "What is the recipe for pasta carbonara?",
    "Who won the FIFA World Cup?",
    "How does quantum entanglement work?",
    "What are the best programming languages?",
    "How do I fix a flat tire?",
    "What is the GDP of France?",
    "How does photosynthesis work?",
    "What are the rules of chess?",
    "What are the treatments for Alzheimer's disease?",
    "How is type 2 diabetes managed?",
]

IN_SCOPE_QUERIES = [
    "What are the warning signs of melanoma?",
    "What are the ABCDE criteria for mole detection?",
    "How does bortezomib treat multiple myeloma?",
    "What causes basal cell carcinoma?",
    "What are survival rates for stage IV melanoma?",
]


def retrieval_relevance_score(question: str, k: int = 3) -> float:
    try:
        docs = semantic_search(
            question,
            k=k,
            mmr=False,
            intent="general",
        )
        if not docs:
            return 0.0

        q_emb = np.asarray(
            embed_model.encode(
                [question],
                normalize_embeddings=True,
                show_progress_bar=False,
            )
        )
        c_emb = np.asarray(
            embed_model.encode(
                [doc["text"] for doc in docs],
                normalize_embeddings=True,
                show_progress_bar=False,
            )
        )
        return float(cosine_similarity(q_emb, c_emb)[0].mean())
    except Exception:
        return 0.0


def _extract_json_object(text: str):
    try:
        return json.loads(text)
    except Exception:
        match = re.search(r"\{.*\}", text or "", flags=re.DOTALL)
        if not match:
            return None
        try:
            return json.loads(match.group(0))
        except Exception:
            return None


def _rule_scope_check(question: str):
    text = question.lower()
    scope_terms = [
        "melanoma",
        "skin cancer",
        "basal cell",
        "squamous cell",
        "multiple myeloma",
        "myeloma",
        "braf",
        "mole",
        "skin lesion",
    ]
    in_scope = any(term in text for term in scope_terms)
    return in_scope, 0.80 if in_scope else 0.80, "rule-based fallback"


def haiku_scope_check(
    question: str,
    retrieval_score: float = None,
):
    if question in OOS_CACHE:
        cached = OOS_CACHE[question]
        return (
            cached["in_scope"],
            cached["confidence"],
            cached["reason"],
        )

    retrieval_score = (
        retrieval_relevance_score(question)
        if retrieval_score is None
        else float(retrieval_score)
    )

    if claude is None:
        return _rule_scope_check(question)

    prompt = f"""
You are the scope gate for a medical RAG assistant.

Allowed scope:
{MEDILINK_SCOPE}

Rules:
- Non-medical questions are out of scope.
- Medical questions about diseases outside this scope are out of scope.
- Do not mark a question in scope merely because it is medical.
- Retrieval similarity is a weak hint only.

Question:
{question}

Retrieval similarity:
{retrieval_score:.3f}

Return only valid JSON:
{{
  "in_scope": true or false,
  "confidence": number from 0 to 1,
  "reason": "short reason"
}}
""".strip()

    try:
        raw = call_haiku(prompt, max_tokens=160)
        data = _extract_json_object(raw)
        if not data:
            return _rule_scope_check(question)

        in_scope = bool(data.get("in_scope", False))
        confidence = max(
            0.0,
            min(1.0, float(data.get("confidence", 0.0))),
        )
        reason = str(data.get("reason", ""))

        OOS_CACHE[question] = {
            "in_scope": in_scope,
            "confidence": confidence,
            "reason": reason,
            "retrieval_score": retrieval_score,
        }
        return in_scope, confidence, reason
    except Exception:
        return _rule_scope_check(question)


def is_in_scope(
    question: str,
    threshold: float = None,
):
    """Return (in_scope, confidence)."""
    threshold = (
        OOS_THRESHOLD
        if threshold is None
        else float(threshold)
    )
    in_scope, confidence, _ = haiku_scope_check(
        question,
        retrieval_relevance_score(question),
    )
    return bool(in_scope and confidence >= threshold), float(confidence)


print("✅ OOS detection ready: is_in_scope()")


## Stage 12 — Cited Answer Generation and Safety Checks
Generates a Sonnet answer, validates PMIDs, and optionally checks grounding.


In [ ]:

# =====================================================================
# 12 — CITED ANSWER GENERATION AND GROUNDING CHECKS
# =====================================================================
MEDICAL_DISCLAIMER = (
    "\n\n⚠️ This information is for educational purposes only and is not "
    "a diagnosis. Consult a qualified healthcare professional for personal care."
)


def verify_citations(answer: str, docs: list) -> dict:
    cited_pmids = re.findall(r"PMID:(\d+)", str(answer))
    available_pmids = {
        str(document.get("pmid"))
        for document in docs
        if document.get("pmid")
    }
    hallucinated = [
        pmid
        for pmid in cited_pmids
        if pmid not in available_pmids
    ]
    return {
        "hallucinated_pmids": hallucinated,
        "cited_count": len(cited_pmids),
        "valid": not hallucinated,
    }


def verify_answer_grounding(answer: str, docs: list) -> str:
    if not USE_HALLUC_CHECK or claude is None or not docs:
        return answer

    context = " | ".join(
        str(document.get("text", ""))[:250]
        for document in docs[:4]
    )

    prompt = f"""
Does this medical answer contain specific claims that are not supported
by the supplied context?

CONTEXT:
{context[:1200]}

ANSWER:
{answer[:1000]}

Reply with exactly GROUNDED or HALLUCINATED.
""".strip()

    try:
        verdict = call_haiku(prompt, max_tokens=15)
    except Exception:
        return answer

    if "HALLUCINATED" in verdict.upper():
        return (
            answer
            + "\n\n⚠️ Some claims may not be fully supported by the "
              "retrieved evidence. Verify them against primary sources."
        )
    return answer


def _prompt_from_existing_docs(
    question: str,
    docs: list,
    intent: str,
) -> str:
    lines = [
        "Answer the question using only the retrieved evidence.",
        "Cite every important medical claim with the exact source identifier.",
        f"Intent: {intent}",
        "",
        "RETRIEVED EXCERPTS:",
    ]

    if not docs:
        lines.append("No relevant evidence was retrieved.")
    else:
        for index, document in enumerate(docs, 1):
            source_id = (
                document.get("pmid")
                or document.get("source")
                or f"doc{index}"
            )
            lines.append(
                f"\n[{index}] PMID:{source_id}\n"
                f"Title: {document.get('title', '')}\n"
                f"Source: {document.get('source', '')}\n"
                f"Text: {document.get('text', '')}"
            )

    lines.extend([
        "",
        f"QUESTION: {question}",
        "",
        "ANSWER:",
    ])
    return "\n".join(lines)


def advanced_rag_answer(
    question: str,
    k: int = None,
    add_disclaimer: bool = True,
    intent: str = None,
    docs: list = None,
    **retrieval_kwargs,
) -> dict:
    """
    Generate one cited answer.

    Pass docs=... to reuse an existing retrieval result and avoid a second
    retrieval pass. When docs is omitted, retrieval is run once here.
    """
    intent = intent or classify_intent(question)

    if docs is None:
        prompt, docs = build_advanced_rag_prompt(
            question,
            k=k or SEARCH_K,
            intent=intent,
            **retrieval_kwargs,
        )
    else:
        docs = list(docs)
        prompt = _prompt_from_existing_docs(
            question,
            docs,
            intent,
        )

    if claude is None:
        answer = (
            "Claude answer generation is disabled. Retrieved evidence is "
            "available in the returned 'docs' field."
        )
    else:
        answer = call_sonnet(
            prompt,
            system=CITATION_SYSTEM,
            history=(
                conversation_history
                if conversation_history
                else None
            ),
        )

    citation_check = verify_citations(answer, docs)
    if citation_check["hallucinated_pmids"]:
        bad_ids = "|".join(
            re.escape(item)
            for item in citation_check["hallucinated_pmids"]
        )
        answer = re.sub(
            rf"PMID:(?:{bad_ids})",
            "PMID:[unverified]",
            answer,
        )

    answer = verify_answer_grounding(answer, docs)

    if add_disclaimer:
        answer += MEDICAL_DISCLAIMER

    update_history("user", question)
    update_history("assistant", answer)

    return {
        "question": question,
        "answer": answer,
        "docs": docs,
        "intent": intent,
        "citation_check": citation_check,
        "pipeline_flags": {
            "hyde": USE_HYDE,
            "multi_query": USE_MULTI_QUERY,
            "cross_encoder": USE_CROSS_ENCODER,
            "compression": USE_COMPRESSION,
            "nli_filter": USE_NLI_FILTER,
            "hallucination_check": USE_HALLUC_CHECK,
            "alpha": _active_alpha(intent),
            "fetch_k": (k or SEARCH_K) * FETCH_K_MULT,
        },
    }


print("✅ Answer generation ready: advanced_rag_answer()")


## Stage 13 — Evaluation Utilities
Defines local metrics and the 30-question benchmark. Nothing runs automatically.


In [ ]:
# LOCAL RAG EVALUATION
# Loads all evaluation functions without running the test automatically.
# Change RUN_EVALUATION to True when you want to execute the benchmark.
# =====================================================================


# ---------------------------------------------------------------------
# TEST PARAMETERS — EDIT THESE
# ---------------------------------------------------------------------

RUN_EVALUATION = False

# Number of chunks retrieved for every test question.
EVAL_SEARCH_K = globals().get("SEARCH_K", 6)

# Choose:
#   "semantic" -> uses semantic_search()
#   "advanced" -> uses advanced_retrieve()
EVAL_RETRIEVAL_MODE = "advanced"

# Used only when EVAL_RETRIEVAL_MODE = "advanced".
EVAL_DISEASE_FILTER = None

# Show retrieval details when advanced_retrieve() supports verbose.
EVAL_VERBOSE = False

# Diversity is printed separately.
# Set to True only if you want it included in the overall score.
INCLUDE_DIVERSITY_IN_OVERALL = False

# Normalize embeddings before cosine similarity.
NORMALIZE_EMBEDDINGS = True


# ---------------------------------------------------------------------
# IMPORTS
# ---------------------------------------------------------------------

import re
import numpy as np

from nltk.tokenize import sent_tokenize
from sklearn.metrics.pairwise import cosine_similarity


# ---------------------------------------------------------------------
# VALIDATION HELPERS
# ---------------------------------------------------------------------

def _check_evaluation_dependencies():
    """
    Check that the required notebook objects exist before evaluation.
    """
    required = [
        "embed_model",
    ]

    if EVAL_RETRIEVAL_MODE == "semantic":
        required.append("semantic_search")
    elif EVAL_RETRIEVAL_MODE == "advanced":
        required.append("advanced_retrieve")
    else:
        raise ValueError(
            "EVAL_RETRIEVAL_MODE must be either "
            "'semantic' or 'advanced'."
        )

    missing = [
        name
        for name in required
        if name not in globals()
    ]

    if missing:
        raise RuntimeError(
            "Evaluation cannot start because these dependencies "
            f"are missing: {', '.join(missing)}"
        )


def _clean_contexts(contexts):
    """
    Remove empty, None, or invalid context values.
    """
    if not contexts:
        return []

    return [
        str(context).strip()
        for context in contexts
        if context is not None and str(context).strip()
    ]


def _safe_sent_tokenize(text):
    """
    Use NLTK sentence tokenization.

    Falls back to a basic regular-expression splitter if the required
    NLTK tokenizer data is unavailable.
    """
    text = str(text).strip()

    if not text:
        return []

    try:
        return [
            sentence.strip()
            for sentence in sent_tokenize(text)
            if sentence.strip()
        ]

    except LookupError:
        return [
            sentence.strip()
            for sentence in re.split(r"(?<=[.!?])\s+", text)
            if sentence.strip()
        ]


def _encode_texts(texts):
    """
    Encode text using the loaded biomedical embedding model.
    """
    texts = [
        str(text).strip()
        for text in texts
        if str(text).strip()
    ]

    if not texts:
        return np.empty((0, 0), dtype=np.float32)

    try:
        return np.asarray(
            embed_model.encode(
                texts,
                normalize_embeddings=NORMALIZE_EMBEDDINGS,
                show_progress_bar=False,
            )
        )

    except TypeError:
        # Compatibility fallback for embedding wrappers that do not
        # accept normalize_embeddings or show_progress_bar.
        embeddings = np.asarray(embed_model.encode(texts))

        if NORMALIZE_EMBEDDINGS:
            norms = np.linalg.norm(
                embeddings,
                axis=1,
                keepdims=True,
            )

            embeddings = embeddings / np.clip(
                norms,
                1e-12,
                None,
            )

        return embeddings


# ---------------------------------------------------------------------
# EVALUATION METRICS
# ---------------------------------------------------------------------

def context_relevance(question: str, contexts: list) -> float:
    """
    Average semantic similarity between the user question and all
    retrieved chunks.
    """
    contexts = _clean_contexts(contexts)

    if not str(question).strip() or not contexts:
        return 0.0

    question_embedding = _encode_texts([question])
    context_embeddings = _encode_texts(contexts)

    if context_embeddings.size == 0:
        return 0.0

    similarities = cosine_similarity(
        question_embedding,
        context_embeddings,
    )[0]

    return float(np.clip(np.mean(similarities), 0.0, 1.0))


def context_coverage(reference_answer: str, contexts: list) -> float:
    """
    Measure whether the retrieved chunks cover the sentences in the
    reference answer.

    Each reference-answer sentence is matched with its most similar
    retrieved chunk.
    """
    contexts = _clean_contexts(contexts)
    answer_sentences = _safe_sent_tokenize(reference_answer)

    if not answer_sentences or not contexts:
        return 0.0

    answer_embeddings = _encode_texts(answer_sentences)
    context_embeddings = _encode_texts(contexts)

    if (
        answer_embeddings.size == 0
        or context_embeddings.size == 0
    ):
        return 0.0

    similarities = cosine_similarity(
        answer_embeddings,
        context_embeddings,
    )

    best_support_per_sentence = similarities.max(axis=1)

    return float(
        np.clip(
            np.mean(best_support_per_sentence),
            0.0,
            1.0,
        )
    )


def answer_faithfulness(reference_answer: str, contexts: list) -> float:
    """
    Semantic support score between the reference answer and retrieved
    chunks.

    Important:
    This is an embedding-based support score, not a complete
    claim-level hallucination or entailment test.
    """
    contexts = _clean_contexts(contexts)

    if not str(reference_answer).strip() or not contexts:
        return 0.0

    answer_embedding = _encode_texts([reference_answer])
    context_embeddings = _encode_texts(contexts)

    if context_embeddings.size == 0:
        return 0.0

    similarities = cosine_similarity(
        answer_embedding,
        context_embeddings,
    )[0]

    return float(np.clip(np.max(similarities), 0.0, 1.0))


def chunk_diversity(contexts: list) -> float:
    """
    Measure how different the retrieved chunks are from each other.

    0.0 means highly repetitive.
    1.0 means highly diverse.
    """
    contexts = _clean_contexts(contexts)

    if not contexts:
        return 0.0

    if len(contexts) == 1:
        return 1.0

    embeddings = _encode_texts(contexts)

    if embeddings.size == 0:
        return 0.0

    similarity_matrix = cosine_similarity(embeddings)

    upper_triangle = similarity_matrix[
        np.triu_indices(
            len(similarity_matrix),
            k=1,
        )
    ]

    if upper_triangle.size == 0:
        return 1.0

    diversity = 1.0 - float(np.mean(upper_triangle))

    return float(np.clip(diversity, 0.0, 1.0))


# ---------------------------------------------------------------------
# RETRIEVAL HELPER
# ---------------------------------------------------------------------

def _retrieve_evaluation_documents(question: str):
    """
    Retrieve documents using the method selected at the top of the cell.
    """
    if EVAL_RETRIEVAL_MODE == "semantic":
        return semantic_search(
            question,
            k=EVAL_SEARCH_K,
        )

    if EVAL_RETRIEVAL_MODE == "advanced":
        kwargs = {
            "k": EVAL_SEARCH_K,
            "disease": EVAL_DISEASE_FILTER,
            "verbose": EVAL_VERBOSE,
        }

        try:
            return advanced_retrieve(
                question,
                **kwargs,
            )

        except TypeError:
            # Compatibility fallback if the current advanced_retrieve()
            # implementation does not accept every optional argument.
            kwargs.pop("verbose", None)

            try:
                return advanced_retrieve(
                    question,
                    **kwargs,
                )

            except TypeError:
                kwargs.pop("disease", None)

                return advanced_retrieve(
                    question,
                    **kwargs,
                )

    raise ValueError(
        f"Unsupported retrieval mode: {EVAL_RETRIEVAL_MODE}"
    )


def _extract_contexts(documents):
    """
    Extract text fields safely from retrieved document dictionaries.
    """
    contexts = []

    for document in documents or []:
        if isinstance(document, dict):
            text = document.get("text", "")
        else:
            text = getattr(document, "text", "")

        if text and str(text).strip():
            contexts.append(str(text).strip())

    return contexts


# ---------------------------------------------------------------------
# TEST QUESTIONS AND REFERENCE ANSWERS
# ---------------------------------------------------------------------

TEST_DATA = [
    {
        "question": (
            "What are the main treatments for multiple myeloma?"
        ),
        "answer": (
            "Multiple myeloma is treated with proteasome inhibitors, "
            "IMiDs, stem cell transplant, and CAR-T cell therapy."
        ),
    },
    {
        "question": (
            "What are immune checkpoint inhibitors in melanoma?"
        ),
        "answer": (
            "Anti-PD1 and anti-CTLA4 antibodies are checkpoint "
            "inhibitors that improve survival in advanced melanoma."
        ),
    },
    {
        "question": "What causes skin cancer?",
        "answer": (
            "UV radiation, genetic mutations in BRAF and NRAS, "
            "and immune suppression are main causes of skin cancer."
        ),
    },
    {
        "question": "What is BRAF mutation role in melanoma?",
        "answer": (
            "BRAF V600E mutation occurs in approximately half of "
            "melanomas and can be targeted by BRAF inhibitors."
        ),
    },
    {
        "question": "What are side effects of CAR-T therapy?",
        "answer": (
            "CAR-T therapy can cause cytokine release syndrome, "
            "neurotoxicity, and prolonged low blood counts."
        ),
    },
]


# ---------------------------------------------------------------------
# MAIN EVALUATION FUNCTION
# ---------------------------------------------------------------------

def run_local_rag_evaluation(test_data=None):
    """
    Run the local retrieval evaluation and return a list of results.

    This function runs only when called manually or when
    RUN_EVALUATION is set to True.
    """
    _check_evaluation_dependencies()

    if test_data is None:
        test_data = TEST_DATA

    if not test_data:
        raise ValueError("No evaluation questions were provided.")

    print("=" * 105)
    print("LOCAL RAG RETRIEVAL EVALUATION")
    print("=" * 105)
    print(f"Retrieval mode : {EVAL_RETRIEVAL_MODE}")
    print(f"Top-K          : {EVAL_SEARCH_K}")
    print(f"Test questions : {len(test_data)}")
    print("=" * 105)
    print()

    all_results = []

    for index, item in enumerate(test_data, start=1):
        question = str(item.get("question", "")).strip()
        reference_answer = str(item.get("answer", "")).strip()

        print(
            f"[{index}/{len(test_data)}] "
            f"Evaluating: {question}"
        )

        try:
            documents = _retrieve_evaluation_documents(question)
            contexts = _extract_contexts(documents)

            result = {
                "question": question,
                "num_contexts": len(contexts),
                "context_relevance": context_relevance(
                    question,
                    contexts,
                ),
                "context_coverage": context_coverage(
                    reference_answer,
                    contexts,
                ),
                "faithfulness": answer_faithfulness(
                    reference_answer,
                    contexts,
                ),
                "chunk_diversity": chunk_diversity(
                    contexts,
                ),
                "error": None,
            }

        except Exception as error:
            result = {
                "question": question,
                "num_contexts": 0,
                "context_relevance": 0.0,
                "context_coverage": 0.0,
                "faithfulness": 0.0,
                "chunk_diversity": 0.0,
                "error": (
                    f"{type(error).__name__}: {error}"
                ),
            }

            print(
                f"  ❌ Failed: "
                f"{type(error).__name__}: {error}"
            )

        all_results.append(result)

    print()
    print(
        f"{'Question':<49} "
        f"{'Docs':>5} "
        f"{'Relevance':>10} "
        f"{'Coverage':>10} "
        f"{'Support':>10} "
        f"{'Diversity':>10}"
    )
    print("─" * 105)

    for result in all_results:
        question_display = result["question"][:47]

        print(
            f"{question_display:<49} "
            f"{result['num_contexts']:>5} "
            f"{result['context_relevance']:>10.3f} "
            f"{result['context_coverage']:>10.3f} "
            f"{result['faithfulness']:>10.3f} "
            f"{result['chunk_diversity']:>10.3f}"
        )

    print("─" * 105)

    relevance_average = float(
        np.mean([
            result["context_relevance"]
            for result in all_results
        ])
    )

    coverage_average = float(
        np.mean([
            result["context_coverage"]
            for result in all_results
        ])
    )

    support_average = float(
        np.mean([
            result["faithfulness"]
            for result in all_results
        ])
    )

    diversity_average = float(
        np.mean([
            result["chunk_diversity"]
            for result in all_results
        ])
    )

    print(
        f"{'AVERAGE':<49} "
        f"{'':>5} "
        f"{relevance_average:>10.3f} "
        f"{coverage_average:>10.3f} "
        f"{support_average:>10.3f} "
        f"{diversity_average:>10.3f}"
    )

    overall_components = [
        relevance_average,
        coverage_average,
        support_average,
    ]

    if INCLUDE_DIVERSITY_IN_OVERALL:
        overall_components.append(diversity_average)

    overall_score = float(np.mean(overall_components))

    print()
    print("── Score Guide ─────────────────────────────────────────")
    print(
        "Context Relevance : average query-to-chunk similarity "
        "(target > 0.60)"
    )
    print(
        "Context Coverage  : reference-answer sentence coverage "
        "(target > 0.60)"
    )
    print(
        "Answer Support    : semantic support from best chunk "
        "(target > 0.70)"
    )
    print(
        "Chunk Diversity   : non-redundancy among chunks "
        "(target > 0.30)"
    )
    print("─────────────────────────────────────────────────────────")

    print(
        f"\n🏆 Overall Retrieval Score: "
        f"{overall_score:.3f} / 1.000"
    )

    if overall_score > 0.75:
        print("   ✅ Excellent — retrieval is performing well.")

    elif overall_score > 0.60:
        print(
            "   ⚠️ Good — consider tuning retrieval parameters."
        )

    else:
        print(
            "   ❌ Needs improvement — inspect retrieved chunks, "
            "chunking, alpha, and reranking."
        )

    errors = [
        result
        for result in all_results
        if result["error"] is not None
    ]

    if errors:
        print(
            f"\n⚠️ {len(errors)} question(s) failed during evaluation."
        )

    summary = {
        "retrieval_mode": EVAL_RETRIEVAL_MODE,
        "search_k": EVAL_SEARCH_K,
        "num_questions": len(all_results),
        "context_relevance": relevance_average,
        "context_coverage": coverage_average,
        "answer_support": support_average,
        "chunk_diversity": diversity_average,
        "overall_score": overall_score,
    }

    return {
        "results": all_results,
        "summary": summary,
    }

# =====================================================================
# CELL 22 — 30-QUESTION DISEASE BENCHMARK
# 3 disease domains x 10 questions each
# Metrics: context_relevance, context_coverage, faithfulness, diversity
# NOTE: diversity is reported separately — NOT included in AVG score.
#       Low diversity is expected in a focused biomedical corpus after
#       MedCPT cross-encoder reranking (semantic consistency = good).
# =====================================================================

BENCHMARK_QUESTIONS = [
    # ── MULTIPLE MYELOMA (10) ────────────────────────────────────────
    {"disease": "multiple myeloma",
     "question": "What are the main treatment options for newly diagnosed multiple myeloma?",
     "ground_truth": "Newly diagnosed myeloma is treated with VRd induction (bortezomib, lenalidomide, dexamethasone), followed by autologous stem cell transplant in eligible patients and lenalidomide maintenance."},
    {"disease": "multiple myeloma",
     "question": "How does bortezomib work as a proteasome inhibitor in myeloma?",
     "ground_truth": "Bortezomib inhibits the 26S proteasome, disrupting protein homeostasis, inducing ER stress, activating apoptosis via caspase cascade, and suppressing NF-kB in myeloma cells."},
    {"disease": "multiple myeloma",
     "question": "What is the role of autologous stem cell transplant in multiple myeloma?",
     "ground_truth": "ASCT consolidates induction response in eligible myeloma patients, deepening remission and improving progression-free and overall survival compared to continuous chemotherapy."},
    {"disease": "multiple myeloma",
     "question": "What are the adverse effects of lenalidomide therapy in myeloma?",
     "ground_truth": "Lenalidomide causes myelosuppression, venous thromboembolism risk, fatigue, diarrhea, rash, and increased risk of secondary primary malignancies."},
    {"disease": "multiple myeloma",
     "question": "How do BCMA-targeting CAR-T therapies work in relapsed multiple myeloma?",
     "ground_truth": "BCMA CAR-T cells kill myeloma via cytotoxic mechanisms, achieving deep remissions including stringent complete responses in heavily pretreated patients."},
    {"disease": "multiple myeloma",
     "question": "What is smoldering multiple myeloma and how does it differ from active disease?",
     "ground_truth": "SMM is an asymptomatic precursor with elevated M-protein and clonal plasma cells but lacking CRAB criteria (hypercalcemia, renal failure, anemia, bone lesions) that define active myeloma."},
    {"disease": "multiple myeloma",
     "question": "What is the prognostic significance of cytogenetic abnormalities in myeloma?",
     "ground_truth": "High-risk cytogenetics including del(17p), t(4;14), and t(14;16) predict shorter PFS and OS, classify patients as R-ISS stage III, and guide treatment intensification."},
    {"disease": "multiple myeloma",
     "question": "How does daratumumab improve outcomes when added to standard myeloma regimens?",
     "ground_truth": "Daratumumab (anti-CD38) depletes myeloma plasma cells and immunosuppressive cells, significantly improving PFS and OS when added to VRd or Rd in newly diagnosed and relapsed myeloma."},
    {"disease": "multiple myeloma",
     "question": "What are bispecific antibodies and their clinical role in relapsed myeloma?",
     "ground_truth": "Bispecific antibodies like teclistamab bind BCMA on myeloma cells and CD3 on T cells, redirecting endogenous T cells to kill myeloma without apheresis, achieving high response rates."},
    {"disease": "multiple myeloma",
     "question": "What staging systems are used in multiple myeloma and what do they measure?",
     "ground_truth": "The R-ISS uses serum beta-2 microglobulin, albumin, LDH, and cytogenetic risk to define stages I-III; higher stages predict shorter survival and guide treatment intensity."},

    # ── MELANOMA (10) ────────────────────────────────────────────────
    {"disease": "melanoma",
     "question": "What is the BRAF V600E mutation and why is it clinically important in melanoma?",
     "ground_truth": "BRAF V600E occurs in ~50% of cutaneous melanomas, constitutively activating MAPK signaling, targeted by vemurafenib and dabrafenib with high response rates."},
    {"disease": "melanoma",
     "question": "How do immune checkpoint inhibitors improve outcomes in advanced melanoma?",
     "ground_truth": "Anti-PD1 agents and anti-CTLA4 reinvigorate exhausted T cells, producing durable responses and 5-year survival rates of 40-52% in advanced melanoma."},
    {"disease": "melanoma",
     "question": "What is the difference between cutaneous melanoma and uveal melanoma?",
     "ground_truth": "Cutaneous melanoma harbors BRAF/NRAS mutations and responds to checkpoint inhibitors; uveal melanoma harbors GNAQ/GNA11 mutations with poor immunotherapy responses."},
    {"disease": "melanoma",
     "question": "What is the role of sentinel lymph node biopsy in melanoma staging?",
     "ground_truth": "SLNB detects microscopic nodal metastasis for staging and guides adjuvant therapy decisions, though completion lymph node dissection does not improve OS."},
    {"disease": "melanoma",
     "question": "How does combined BRAF plus MEK inhibition improve outcomes over BRAF inhibition alone?",
     "ground_truth": "Adding MEK inhibitors to BRAF inhibitors blocks paradoxical MAPK reactivation, delays acquired resistance, reduces cutaneous toxicity, and improves PFS and OS."},
    {"disease": "melanoma",
     "question": "What clinical and molecular factors predict response to anti-PD1 therapy in melanoma?",
     "ground_truth": "Higher tumor mutational burden, PD-L1 expression, tumor-infiltrating lymphocytes, PTEN intact status, and microsatellite instability predict anti-PD1 response."},
    {"disease": "melanoma",
     "question": "What are the immune-related adverse events of checkpoint inhibitors in melanoma?",
     "ground_truth": "irAEs include colitis, hepatitis, pneumonitis, endocrinopathies such as hypothyroidism and hypophysitis, nephritis, and skin toxicities, managed with corticosteroids."},
    {"disease": "melanoma",
     "question": "What is the prognosis of stage IV melanoma with modern immunotherapy?",
     "ground_truth": "With nivolumab plus ipilimumab combination, 5-year OS reaches ~52% in stage IV melanoma, up from historical rates below 10% with chemotherapy."},
    {"disease": "melanoma",
     "question": "How is adjuvant therapy used after resection of stage III melanoma?",
     "ground_truth": "Adjuvant anti-PD1 or BRAF+MEK inhibitors in BRAF-mutant patients significantly reduce recurrence and improve relapse-free survival after complete resection."},
    {"disease": "melanoma",
     "question": "What are the molecular subtypes of melanoma beyond BRAF mutation?",
     "ground_truth": "Molecular subtypes include BRAF-mutant (~50%), NRAS-mutant (~25%), NF1-mutant (~14%), and triple-wild-type (~11%), each with distinct pathway activation and therapeutic vulnerabilities."},

    # ── SKIN CANCER (10) ─────────────────────────────────────────────
    {"disease": "skin cancer",
     "question": "What are the main types of non-melanoma skin cancer and how do they differ?",
     "ground_truth": "Basal cell carcinoma arises from basal keratinocytes with rare metastasis; squamous cell carcinoma from spinous keratinocytes with higher metastatic potential; both are UV-driven."},
    {"disease": "skin cancer",
     "question": "How is cutaneous squamous cell carcinoma treated in clinical practice?",
     "ground_truth": "SCC is treated with surgical excision, Mohs surgery for high-risk sites, radiation for inoperable cases, and cemiplimab immunotherapy for locally advanced or metastatic disease."},
    {"disease": "skin cancer",
     "question": "What is Mohs micrographic surgery and when is it indicated for skin cancer?",
     "ground_truth": "Mohs surgery removes thin layers with real-time microscopic margin assessment until clear, indicated for high-risk BCC/SCC at cosmetically sensitive or functionally critical sites."},
    {"disease": "skin cancer",
     "question": "What are the risk factors for developing basal cell carcinoma?",
     "ground_truth": "BCC risk factors include cumulative UV radiation, fair skin, immunosuppression, prior radiation, arsenic exposure, and Gorlin syndrome (PTCH1 mutations)."},
    {"disease": "skin cancer",
     "question": "How does UV radiation cause DNA damage that leads to skin cancer?",
     "ground_truth": "UV-B induces cyclobutane pyrimidine dimers causing C-to-T and CC-to-TT transition mutations (UV signature) in tumor suppressors TP53 and CDKN2A."},
    {"disease": "skin cancer",
     "question": "What is the role of the hedgehog signaling pathway in basal cell carcinoma?",
     "ground_truth": "PTCH1 loss or SMO gain-of-function mutations constitutively activate GLI transcription factors in most BCCs; vismodegib and sonidegib target this pathway."},
    {"disease": "skin cancer",
     "question": "What is the prognosis and treatment approach for Merkel cell carcinoma?",
     "ground_truth": "MCC is a rare aggressive neuroendocrine skin cancer; avelumab and pembrolizumab produce durable responses in metastatic disease previously associated with poor prognosis."},
    {"disease": "skin cancer",
     "question": "How is actinic keratosis managed to prevent progression to squamous cell carcinoma?",
     "ground_truth": "AKs are treated with topical 5-FU, imiquimod, or diclofenac; photodynamic therapy; or cryotherapy to eliminate dysplastic keratinocytes before malignant transformation."},
    {"disease": "skin cancer",
     "question": "What genetic syndromes predispose individuals to multiple skin cancers?",
     "ground_truth": "Xeroderma pigmentosum (NER deficiency) causes multiple skin cancers; Gorlin syndrome (PTCH1) causes multiple BCCs; familial melanoma involves CDKN2A germline mutations."},
    {"disease": "skin cancer",
     "question": "How does immunosuppression after organ transplantation increase skin cancer risk?",
     "ground_truth": "Transplant immunosuppression impairs T-cell surveillance, increases carcinogenic HPV persistence, and azathioprine has direct mutagenic effects, causing 65-250x elevated SCC risk."},
]


def run_benchmark(benchmark, use_advanced=True, verbose=True):
    """
    Run full 30-question benchmark.
    AVG = mean(relevance, coverage, faithfulness) only.
    Diversity is measured and displayed but excluded from AVG —
    low diversity is expected in focused biomedical retrieval after
    cross-encoder reranking and is not a quality defect.
    """
    results = []
    for item in benchmark:
        q  = item["question"]
        gt = item["ground_truth"]

        if use_advanced and "advanced_retrieve" in dir():
            docs = advanced_retrieve(q, k=SEARCH_K, disease=item["disease"], verbose=False)
        else:
            docs = semantic_search(q, k=SEARCH_K)

        contexts = [d["text"] for d in docs]
        rel   = context_relevance(q,  contexts)
        cov   = context_coverage(gt,  contexts)
        faith = answer_faithfulness(gt, contexts)
        div   = chunk_diversity(contexts)

        # ── FIX: diversity excluded from avg (domain-specific corpus characteristic) ──
        avg = float(np.mean([rel, cov, faith]))

        result = {
            "disease":   item["disease"],
            "question":  q[:65],
            "relevance": round(rel,   3),
            "coverage":  round(cov,   3),
            "faithful":  round(faith, 3),
            "diversity": round(div,   3),
            "avg":       round(avg,   3),
        }
        results.append(result)
        if verbose:
            print(f"  {result['disease']:<22} {avg:.3f}  {q[:50]}")
    return results

print("✅ Evaluation utilities ready; no benchmark was run automatically.")


## Stage 14 — Medical Triage
Collects safe clarification details for personal symptom descriptions.


In [ ]:
# =====================================================================
# CELL 27 — HAIKU-POWERED MEDICAL TRIAGE LAYER
# =====================================================================
# Haiku decides:
#   1. Is this a personal symptom intake?
#   2. Should triage be shown?
#   3. What follow-up questions are needed?
#   4. How to convert answers into a clinical RAG query
# =====================================================================

import ipywidgets as w
from IPython.display import display, HTML, clear_output
import json, re, time

# ---------------------------------------------------------------------
# Safety / fallback triage flows
# ---------------------------------------------------------------------
DEFAULT_TRIAGE_FLOWS = {
    "skin_lesion": {
        "title": "Skin lesion / mole triage",
        "questions": [
            {
                "key": "location",
                "question": "Where is it on your body?",
                "options": ["Face/neck", "Arm/hand", "Leg/foot", "Back/chest", "Other"]
            },
            {
                "key": "duration",
                "question": "How long have you noticed it?",
                "options": ["Just noticed it", "Few weeks", "Few months", "Over a year"]
            },
            {
                "key": "change",
                "question": "Has it changed recently?",
                "options": ["Size changed", "Color changed", "Shape changed", "Multiple changes", "No clear change"]
            },
            {
                "key": "symptoms",
                "question": "Any symptoms from it?",
                "options": ["Itching", "Bleeding", "Pain", "Crusting/oozing", "None"]
            }
        ],
        "query_template": "melanoma warning signs skin lesion mole {location} {duration} {change} {symptoms} ABCDE ugly duckling bleeding itching diagnosis"
    },
    "systemic_myeloma": {
        "title": "Possible systemic / myeloma-related symptoms",
        "questions": [
            {
                "key": "pain",
                "question": "Do you have bone or back pain?",
                "options": ["Severe", "Mild/moderate", "No"]
            },
            {
                "key": "fatigue",
                "question": "Do you feel unusually tired or weak?",
                "options": ["Yes, severe", "Yes, mild", "No"]
            },
            {
                "key": "infections",
                "question": "Have you had frequent infections recently?",
                "options": ["Yes, multiple", "Yes, one or two", "No"]
            },
            {
                "key": "other",
                "question": "Any other symptoms?",
                "options": ["Weight loss", "Night sweats", "Numbness/tingling", "None"]
            }
        ],
        "query_template": "multiple myeloma symptoms {pain} {fatigue} {infections} {other} bone pain anemia infections diagnosis"
    }
}


def _extract_json_from_text(text: str):
    """
    Robust JSON extractor for Haiku responses.
    """
    if not text:
        return None

    text = text.strip()

    try:
        return json.loads(text)
    except Exception:
        pass

    # Extract JSON object between first { and last }
    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not m:
        return None

    try:
        return json.loads(m.group(0))
    except Exception:
        return None


def _fallback_triage_decision(user_text: str):
    """
    Fallback if Haiku is unavailable or returns invalid JSON.
    """
    q = user_text.lower()

    educational_starters = [
        "what is", "what are", "explain", "tell me", "define",
        "how is", "how do", "causes of", "treatment for",
        "symptoms of", "signs of"
    ]

    if any(q.startswith(s) for s in educational_starters):
        return {
            "needs_triage": False,
            "reason": "Educational/general medical question.",
            "flow_type": "none",
            "questions": [],
            "query_template": ""
        }

    skin_terms = [
        "my mole", "my skin", "my spot", "i have a mole", "i noticed a mole",
        "dark spot", "black spot", "brown spot", "changed color", "bleeding mole",
        "itchy mole", "new mole", "skin lesion", "spot changed", "mole changed"
    ]

    myeloma_terms = [
        "bone pain", "back pain", "fatigue", "tired all the time",
        "weak", "frequent infections", "weight loss", "night sweats",
        "numbness", "tingling"
    ]

    if any(t in q for t in skin_terms):
        flow = DEFAULT_TRIAGE_FLOWS["skin_lesion"]
        return {
            "needs_triage": True,
            "reason": "Personal skin symptom intake.",
            "flow_type": "skin_lesion",
            "questions": flow["questions"],
            "query_template": flow["query_template"]
        }

    if any(t in q for t in myeloma_terms):
        flow = DEFAULT_TRIAGE_FLOWS["systemic_myeloma"]
        return {
            "needs_triage": True,
            "reason": "Personal systemic symptom intake.",
            "flow_type": "systemic_myeloma",
            "questions": flow["questions"],
            "query_template": flow["query_template"]
        }

    return {
        "needs_triage": False,
        "reason": "No personal symptom intake detected.",
        "flow_type": "none",
        "questions": [],
        "query_template": ""
    }


def haiku_triage_decision(user_text: str):
    """
    Ask Haiku whether triage is needed and what questions to ask.
    Returns structured dict.
    """
    # If call_haiku is not loaded/configured, use fallback
    if "call_haiku" not in globals() or globals().get("claude", None) is None:
        return _fallback_triage_decision(user_text)

    prompt = f"""
You are a medical triage router for an educational RAG assistant about:
- melanoma
- skin cancer
- multiple myeloma

Decide whether the user message is a PERSONAL SYMPTOM INTAKE requiring triage questions.

Important:
- If the user asks a general educational question like "What are the warning signs of melanoma?", triage is NOT needed.
- If the user says "I have...", "my mole...", "my skin...", "I noticed...", "I feel...", triage IS needed.
- Do NOT diagnose.
- Ask only safe clarifying questions.
- Return JSON only.

User message:
{user_text}

Return exactly this JSON structure:
{{
  "needs_triage": true or false,
  "reason": "short reason",
  "flow_type": "skin_lesion" or "systemic_myeloma" or "urgent" or "none",
  "questions": [
    {{
      "key": "short_key",
      "question": "question to ask user",
      "options": ["option1", "option2", "option3", "option4"]
    }}
  ],
  "query_template": "clinical retrieval query with placeholders like {{short_key}}"
}}

Rules:
- If skin/mole/spot/lesion issue: ask about location, duration, changes, bleeding/itching/pain.
- If fatigue/bone pain/infections/weight loss: ask about bone pain, fatigue severity, infections, weight loss/night sweats/numbness.
- If urgent red flags like severe bleeding, trouble breathing, stroke, chest pain: flow_type="urgent".
- For educational questions, needs_triage=false and questions=[].
"""

    try:
        raw = call_haiku(prompt, max_tokens=900)
        data = _extract_json_from_text(raw)

        if not isinstance(data, dict):
            return _fallback_triage_decision(user_text)

        # Basic validation
        data.setdefault("needs_triage", False)
        data.setdefault("reason", "")
        data.setdefault("flow_type", "none")
        data.setdefault("questions", [])
        data.setdefault("query_template", "")

        if not isinstance(data["questions"], list):
            data["questions"] = []

        return data

    except Exception as e:
        print(f"⚠️ Haiku triage failed, using fallback: {e}")
        return _fallback_triage_decision(user_text)


def build_haiku_clinical_query(original_question: str, decision: dict, answers: dict):
    """
    Ask Haiku to convert triage answers into a clinical retrieval query.
    Fallback uses query_template.
    """
    template = decision.get("query_template", "")

    # Fallback template fill
    fallback_query = template
    for k, v in answers.items():
        fallback_query = fallback_query.replace("{" + k + "}", str(v).lower())

    fallback_query = re.sub(r"\{\w+\}", "", fallback_query)
    fallback_query = re.sub(r"\s+", " ", fallback_query).strip()

    if not fallback_query:
        fallback_query = original_question + " " + " ".join(str(v) for v in answers.values())

    if "call_haiku" not in globals() or globals().get("claude", None) is None:
        return fallback_query

    prompt = f"""
Convert this symptom triage into a concise biomedical retrieval query for a RAG search.

Original user message:
{original_question}

Triage type:
{decision.get("flow_type")}

Answers:
{json.dumps(answers, ensure_ascii=False)}

Return only the retrieval query.
Do not diagnose.
Include useful biomedical terms, e.g. melanoma ABCDE warning signs, ugly duckling, bleeding, itching, skin lesion, or multiple myeloma symptoms if relevant.
"""

    try:
        query = call_haiku(prompt, max_tokens=120).strip()
        query = query.replace("\n", " ").strip()

        if len(query) < 5:
            return fallback_query

        return query

    except Exception:
        return fallback_query


class HaikuTriageWidget:
    """
    Dynamic triage widget using Haiku-generated questions.
    """
    def __init__(self, question, decision, on_complete):
        self.question = question
        self.decision = decision
        self.questions = decision.get("questions", [])
        self.on_complete = on_complete
        self.answers = {}
        self.step = 0

        self.container = w.VBox(
            layout=w.Layout(
                border="1px solid #1e3a5a",
                padding="0",
                max_width="760px"
            )
        )

        self.out = w.Output()
        self.container.children = [self.out]
        self._render_step()

    def _safe_questions(self):
        clean = []

        for i, q in enumerate(self.questions):
            if not isinstance(q, dict):
                continue

            key = q.get("key", f"q{i+1}")
            question = q.get("question", "Can you clarify?")
            options = q.get("options", [])

            if not isinstance(options, list) or len(options) < 2:
                options = ["Yes", "No", "Not sure"]

            clean.append({
                "key": key,
                "question": question,
                "options": [str(x) for x in options[:6]]
            })

        # Fallback if Haiku gave no good questions
        if not clean:
            fallback = DEFAULT_TRIAGE_FLOWS["skin_lesion"]
            clean = fallback["questions"]
            self.decision["query_template"] = fallback["query_template"]

        return clean

    def _render_step(self):
        questions = self._safe_questions()

        with self.out:
            clear_output(wait=True)

            total = len(questions)

            if self.step >= total:
                clinical_q = build_haiku_clinical_query(
                    self.question,
                    self.decision,
                    self.answers
                )

                display(HTML(f"""
                <div style="background:#1a1a2e;color:#e0e0ff;padding:14px 20px;
                            border-radius:10px 10px 0 0;font-size:15px;font-weight:700;">
                  🩺 Haiku Triage Complete
                </div>
                <div style="background:#0f3460;padding:12px 16px;border-radius:0 0 10px 10px;">
                  <div style="color:#80deea;font-size:12px;margin-bottom:8px;">
                    Original: <i>{self.question}</i>
                  </div>
                  <div style="color:#a5d6a7;font-size:13px;margin-bottom:8px;">
                    Reason: {self.decision.get("reason", "")}
                  </div>
                  <div style="color:#ffe082;font-size:13px;margin-bottom:12px;">
                    Clinical query: <b>{clinical_q}</b>
                  </div>
                  <div style="color:#ffe082;font-size:12px;">
                    ⏳ Searching medical literature…
                  </div>
                </div>
                """))

                self.on_complete(clinical_q, self.answers, self.decision)
                return

            q = questions[self.step]
            key = q["key"]
            question_text = q["question"]
            options = q["options"]

            display(HTML(f"""
            <div style="background:#1a1a2e;color:#e0e0ff;padding:14px 20px;
                        border-radius:10px 10px 0 0;font-size:15px;font-weight:700;">
              🩺 Haiku Medical Triage
              <span style="font-size:11px;font-weight:400;color:#80deea;">
                Step {self.step+1} of {total}
              </span>
            </div>
            <div style="background:#0f3460;padding:12px 16px;">
              <div style="color:#80deea;font-size:12px;margin-bottom:6px;">
                You said: <i>{self.question}</i>
              </div>
              <div style="color:#a5d6a7;font-size:12px;margin-bottom:6px;">
                Haiku decision: {self.decision.get("flow_type", "unknown")} — {self.decision.get("reason", "")}
              </div>
              <div style="color:#ffe082;font-size:14px;font-weight:600;margin-bottom:10px;">
                {question_text}
              </div>
            </div>
            """))

            buttons = []

            for opt in options:
                btn = w.Button(
                    description=str(opt),
                    layout=w.Layout(width="auto", height="36px", margin="4px"),
                    style={"button_color": "#0f3460"},
                )
                btn.on_click(lambda b, k=key, v=opt: self._answer(k, v))
                buttons.append(btn)

            display(w.HBox(buttons, layout=w.Layout(flex_wrap="wrap", padding="8px 16px")))

    def _answer(self, key, value):
        self.answers[key] = value
        self.step += 1
        self._render_step()

    def display(self):
        display(self.container)


def run_triage(question, on_result=None):
    """
    Main function used by Control Panel.
    Haiku decides whether triage is needed.
    Returns True if triage UI was shown, False if direct RAG should run.
    """
    decision = haiku_triage_decision(question)

    if not decision.get("needs_triage", False):
        return False

    if decision.get("flow_type") == "urgent":
        display(HTML("""
        <div style="background:#3b0d0d;color:#ffb3b3;padding:14px;border-radius:8px;">
          🚨 This may need urgent medical attention. If symptoms are severe or rapidly worsening,
          contact emergency services or seek urgent care immediately.
        </div>
        """))
        return True

    def _complete(clinical_q, answers, decision_obj):
        if on_result:
            on_result(clinical_q)

    widget = HaikuTriageWidget(question, decision, _complete)
    widget.display()
    return True


# Backward-compatible names used by Cell 24
def detect_triage_flow(question):
    decision = haiku_triage_decision(question)
    if decision.get("needs_triage", False):
        return decision.get("flow_type", "triage")
    return None


TriageWidget = HaikuTriageWidget

print("✅ Medical triage ready: run_triage()")


## Stage 15 — Master Router
Routes greetings, emergencies, personal symptoms, medical questions, and multimodal requests.


In [ ]:
# =====================================================================
# CELL 22 — HAIKU MASTER ROUTER (25 PATHS)
# =====================================================================

EMERGENCY_KEYWORDS = ["chest pain","can't breathe","heart attack","stroke",
    "unconscious","not breathing","severe bleeding","overdose","seizure"]
CRISIS_KEYWORDS    = ["want to hurt myself","want to die","kill myself","suicide",
    "self harm","self-harm","end my life","no reason to live"]
MEDICAL_DISCLAIMER = ("\n\n⚠️ *Educational only. Consult a healthcare provider.*")
EMERGENCY_RESPONSE = ("🚨 **Medical Emergency.**\n\nCall your local emergency services or go to the nearest emergency department immediately.")
CRISIS_RESPONSE = (
    "I am concerned about your safety. Contact local emergency or crisis "
    "services now, and stay with a trusted person if possible."
)

def _is_emergency(t): return any(k in t.lower() for k in EMERGENCY_KEYWORDS)
def _is_crisis(t):    return any(k in t.lower() for k in CRISIS_KEYWORDS)

def _classify_router_intent(text):
    if not claude: return "medical_question"
    return call_haiku(
        f"""Classify for medical chatbot:
- empty: no content
- greeting: hi/hello/how are you
- thanks: thank you/thanks
- followup: tell me more/continue/what about
- emergency: chest pain/heart attack/stroke (life-threatening)
- crisis: self-harm/suicide/want to die
- urgent_symptom: severe pain needing same-day care
- symptom_description: personal symptoms (my skin, I have a mole, I feel tired)
- medical_question: disease/treatment/medication facts
- medication_question: specific drug/dose/interaction
- second_opinion: "my doctor said" or evaluate advice
- non_medical: weather/sports/recipes
- out_of_scope: clearly non-medical

Reply ONE word.
Input: {text}""", max_tokens=20)

def _rag_answer_full(question, use_disclaimer=True, intent=None):
    result = advanced_rag_answer(question, intent=intent, add_disclaimer=use_disclaimer)
    return result["answer"], result["docs"]


def _retrieve_only(question, intent=None, k=None):
    """Retrieve evidence without calling Claude/Sonnet final answer."""
    if intent is None:
        intent = classify_intent(question) if "classify_intent" in globals() else "general"
    docs = advanced_retrieve(
        question,
        k=k or globals().get("SEARCH_K", 6),
        disease=globals().get("DISEASE_FILTER", None),
        intent=intent,
        use_nli_filter=False,
        verbose=False,
    )
    return docs


def route(question="", image=None, image_type=None, run_mode="answer"):
    result = {"answer":"","path":"","docs":[],"triage_triggered":False,"emergency":False}
    q = question.strip()
    retrieve_only = str(run_mode).lower().startswith("retrieve")

    if not q and image is None:
        result["answer"] = "Hello! How can I help you today? I specialize in melanoma, skin cancer, and multiple myeloma."
        result["path"]   = "empty"
        return result

    if q and _is_crisis(q):
        result.update({"answer":CRISIS_RESPONSE,"path":"crisis","emergency":True}); return result
    if q and _is_emergency(q) and not image:
        result.update({"answer":EMERGENCY_RESPONSE,"path":"emergency","emergency":True}); return result

    if image and not q:
        if image_type == "blurry":
            result["answer"] = "The image is unclear. Could you send a clearer photo?"; result["path"] = "blurry_image"
        elif image_type == "non_medical":
            result["answer"] = "I can only analyze medical images like skin lesions or moles."; result["path"] = "non_medical_image"
        elif image_type == "medical_document":
            result["answer"] = call_sonnet("Explain this medical document in simple language for a patient.")
            result["path"] = "medical_document"
        else:
            try:
                cv_result = run_cv_model(image, image_type=image_type or "skin_lesion")
                result["answer"] = interpret_cv_result(cv_result)
                result["cv_result"] = cv_result
                result["path"] = "medical_image_only_htan"
            except Exception as e:
                result["answer"] = f"HTAN image analysis failed: {e}"
                result["path"] = "medical_image_htan_error"
        return result

    if image and q:
        if _is_emergency(q):
            result.update({"answer":EMERGENCY_RESPONSE,"path":"medical_image_urgent","emergency":True}); return result
        cv_result = None
        cv_note = ""
        if image_type != "non_medical":
            try:
                cv_result = run_cv_model(image, image_type=image_type or "skin_lesion")
                cv_note = interpret_cv_result(cv_result, question=q) + "\n\n"
            except Exception as e:
                cv_note = f"HTAN image analysis failed: {e}\n\n"
        if image_type == "non_medical":
            intent = _classify_router_intent(q)
            if "medical" in intent:
                a, docs = _rag_answer_full(q)
                result.update({"answer":f"*Image not medically relevant — answering question:*\n\n{a}","docs":docs,"path":"non_medical_img_medical_text"})
            else:
                result.update({"answer":"I can only help with medical questions.","path":"non_medical_both"})
        else:
            a, docs = _rag_answer_full(q)
            if cv_result is not None:
                cv_note = interpret_cv_result(cv_result, question=q, docs=docs) + "\n\n"
            result.update({"answer":cv_note+a,"docs":docs,"cv_result":cv_result,"path":"medical_image_medical_text_htan"})
        return result

    intent_label = _classify_router_intent(q)

    handlers = {
        "empty":    lambda: ("How can I help you today?", "empty"),
        "greeting": lambda: (call_haiku(f"User said: '{q}'. Respond warmly as medical AI.", max_tokens=100), "greeting"),
        "thanks":   lambda: (call_haiku(f"User said: '{q}'. Respond naturally, offer help.", max_tokens=80), "thanks"),
        "followup": lambda: (call_haiku(q, max_tokens=300, history=conversation_history), "followup"),
        "crisis":   lambda: (CRISIS_RESPONSE, "crisis"),
        "emergency":lambda: (EMERGENCY_RESPONSE, "emergency"),
        "urgent_symptom": lambda: (EMERGENCY_RESPONSE+"\n\n*If not life-threatening, see a doctor today.*", "urgent"),
        "non_medical":    lambda: (call_haiku(f"Answer helpfully: {q}", max_tokens=200), "non_medical"),
        "out_of_scope":   lambda: (call_haiku(f"Politely decline: user asked: '{q}'. You specialize in skin cancer + myeloma only.", max_tokens=100), "out_of_scope"),
    }

    if intent_label in handlers:
        answer, path = handlers[intent_label]()
        result.update({"answer":answer,"path":path})
        if intent_label in ("crisis","emergency"): result["emergency"] = True
        update_history("user", q); update_history("assistant", answer)
        return result

    if intent_label == "symptom_description":
        flow = detect_triage_flow(q)
        if flow:
            result.update({"triage_triggered":True,"path":f"triage_{flow}","answer":f"__TRIAGE__{flow}"})
            return result
        rewritten = rewrite_query(q, intent="symptoms")
        a, docs = _rag_answer_full(rewritten, intent="symptoms")
        result.update({"answer":a+MEDICAL_DISCLAIMER,"docs":docs,"path":"symptom_direct"})

    elif intent_label in ("medical_question","medication_question","second_opinion"):
        intent = classify_intent(q)
        rewritten = rewrite_query(q, intent=intent)
        in_scope, score = is_in_scope(rewritten)
        if not in_scope:
            result.update({"answer":f"I don't have relevant information on that topic (score={score:.2f}). I specialize in melanoma, skin cancer, and multiple myeloma.","path":"oos_blocked"})
        else:
            a, docs = _rag_answer_full(rewritten, intent=intent,
                use_disclaimer=(intent_label in ("medication_question","second_opinion")))
            result.update({"answer":a,"docs":docs,"path":f"medical_rag_{intent_label}"})
    else:
        a, docs = _rag_answer_full(q)
        result.update({"answer":a,"docs":docs,"path":"medical_rag_general"})

    if not result["triage_triggered"]:
        update_history("user", q)
        update_history("assistant", result["answer"])

    return result

print("✅ Cell 22 ready — route() master router (25 paths)")


## Stage 16 — HTAN Image Integration
Runs lesion segmentation and converts structured image output into retrieval context.


In [ ]:
# =====================================================================
# CELL 23 — HTAN CV MODEL INTEGRATION — CLEAN / SAFE
# =====================================================================
# Purpose:
#   ✅ Runs HTAN image segmentation
#   ✅ Returns structured model output for retrieval augmentation
#   ✅ Provides local non-Claude summaries for retrieve-only mode
#   ✅ Does NOT diagnose from image alone
# =====================================================================

import os, sys, json, tempfile
from pathlib import Path

HTAN_ROOT_CANDIDATES = [
    Path(os.environ.get("HTAN_ROOT", "")).expanduser() if os.environ.get("HTAN_ROOT") else None,
    Path("/home/ubuntu/HTAN"),
    Path.cwd() / "HTAN",
]
HTAN_ROOT = next((p for p in HTAN_ROOT_CANDIDATES if p and p.exists()), Path.cwd() / "HTAN")
HTAN_CHECKPOINT = HTAN_ROOT / "saves" / "htan_2_n2" / "best_model.pth"
HTAN_PIXEL_SPACING_MM = globals().get("HTAN_PIXEL_SPACING_MM", 0.1)

if str(HTAN_ROOT) not in sys.path:
    sys.path.insert(0, str(HTAN_ROOT))

_HTAN_MODEL_CACHE = {}


def _image_to_path(image_data):
    """Accept a path, PIL image, numpy array, or bytes and return an image file path."""
    if image_data is None:
        raise ValueError("No image data was provided.")

    if isinstance(image_data, (str, os.PathLike)):
        image_path = Path(image_data).expanduser()
        if not image_path.exists():
            raise FileNotFoundError(f"Image not found: {image_path}")
        return str(image_path)

    suffix = ".png"
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=suffix)
    tmp.close()

    if isinstance(image_data, bytes):
        Path(tmp.name).write_bytes(image_data)
        return tmp.name

    try:
        from PIL import Image
        import numpy as np

        if isinstance(image_data, Image.Image):
            image_data.convert("RGB").save(tmp.name)
            return tmp.name

        if isinstance(image_data, np.ndarray):
            arr = image_data
            if arr.dtype != np.uint8:
                arr = np.clip(arr, 0, 255).astype(np.uint8)
            Image.fromarray(arr).convert("RGB").save(tmp.name)
            return tmp.name
    except Exception:
        pass

    raise TypeError("Unsupported image input. Use a file path, PIL image, NumPy array, or bytes.")


def _load_htan_model(checkpoint_path=HTAN_CHECKPOINT):
    checkpoint_path = str(Path(checkpoint_path).expanduser())
    if checkpoint_path not in _HTAN_MODEL_CACHE:
        if not Path(checkpoint_path).exists():
            raise FileNotFoundError(
                f"HTAN weights not found at {checkpoint_path}. Run the Hugging Face download cell first."
            )
        from inference import load_model
        _HTAN_MODEL_CACHE[checkpoint_path] = load_model(checkpoint_path)
    return _HTAN_MODEL_CACHE[checkpoint_path]


def run_cv_model(image_data, image_type="skin_lesion", pixel_spacing_mm=None, checkpoint_path=None):
    """Run HTAN segmentation and return MediLink-compatible CV output."""
    from inference import segment

    checkpoint_path = Path(checkpoint_path or HTAN_CHECKPOINT)
    pixel_spacing_mm = HTAN_PIXEL_SPACING_MM if pixel_spacing_mm is None else pixel_spacing_mm
    image_path = _image_to_path(image_data)
    model = _load_htan_model(checkpoint_path)

    result = segment(
        image_path=image_path,
        checkpoint_path=str(checkpoint_path),
        pixel_spacing_mm=pixel_spacing_mm,
        model=model,
    )

    predicted_class = "lesion_detected" if result.get("tumor_detected") else "no_lesion_detected"

    return {
        "segmentation_mask": result.get("binary_mask"),
        "confidence": result.get("confidence_score", 0.0),
        "predicted_class": predicted_class,
        "features": {
            # Keep original keys for compatibility with existing code.
            "tumor_detected": result.get("tumor_detected", False),
            "tumor_area_percent": result.get("tumor_area_percent", 0.0),
            "tumor_area_cm2": result.get("tumor_area_cm2", 0.0),
            "num_lesions": result.get("num_lesions", 0),
            "largest_lesion_diameter_mm": result.get("largest_lesion_diameter_mm", 0.0),
            "lesion_location": result.get("lesion_location", "unknown"),
            "severity_estimate": result.get("severity_estimate", "unknown"),
        },
        "htan_result": result,
        "image_path": image_path,
        "model": result.get("model", "HTAN_2_n2"),
    }


def _cv_summary_for_llm(cv_result):
    """Compact JSON summary that is safe to pass into an LLM prompt."""
    if not cv_result:
        return "{}"

    htan = cv_result.get("htan_result", {})
    if htan:
        summary = {k: v for k, v in htan.items() if k not in ("probability_map", "binary_mask")}
    else:
        summary = {
            "lesion_detected": cv_result.get("features", {}).get("tumor_detected", False),
            "confidence_score": cv_result.get("confidence", 0.0),
            **cv_result.get("features", {}),
        }

    return json.dumps(summary, indent=2, default=str)


def cv_result_to_retrieval_text(cv_result):
    """
    Converts HTAN output to text that can augment the retrieval query.
    This is NOT a diagnosis. It only helps retrieval focus on visual findings.
    """
    f = cv_result.get("features", {}) if cv_result else {}
    conf = cv_result.get("confidence", 0.0) if cv_result else 0.0

    return (
        "HTAN image segmentation structured output: "
        f"lesion_detected={f.get('tumor_detected', False)}; "
        f"confidence={conf:.3f}; "
        f"tumor_area_percent={f.get('tumor_area_percent', 0)}; "
        f"num_lesions={f.get('num_lesions', 0)}; "
        f"largest_lesion_diameter_mm={f.get('largest_lesion_diameter_mm', 0)}; "
        f"lesion_location={f.get('lesion_location', 'unknown')}; "
        f"screening_priority={f.get('severity_estimate', 'unknown')}."
    )


def format_cv_local_summary(cv_result, question="", mode="retrieve_only"):
    """
    Local HTAN summary. Does NOT call Claude.
    Use in retrieve-only mode and as structured intro before cited RAG answer.
    """
    f = cv_result.get("features", {}) if cv_result else {}
    detected = f.get("tumor_detected", False)
    confidence = cv_result.get("confidence", 0.0) if cv_result else 0.0

    if mode == "retrieve_answer":
        header = (
            "HTAN structured image analysis used for RAG context.\n"
            "A cited diagnostic-support summary is generated below using retrieved evidence.\n"
        )
    else:
        header = (
            "HTAN image segmentation output only — no generated medical explanation.\n"
            "The model output was used to retrieve RAG evidence below.\n"
        )

    lines = [
        header,
        "Model output:",
        f"- Lesion detected: {detected}",
        f"- Confidence: {confidence:.1%}",
        f"- Area: {f.get('tumor_area_percent', 0)}%",
        f"- Area cm²: {f.get('tumor_area_cm2', 0)}",
        f"- Number of lesions: {f.get('num_lesions', 0)}",
        f"- Largest diameter: {f.get('largest_lesion_diameter_mm', 0)} mm",
        f"- Location: {f.get('lesion_location', 'unknown')}",
        f"- Screening priority: {f.get('severity_estimate', 'unknown')}",
        "",
        "This is segmentation/screening support only, not a diagnosis.",
    ]

    if question:
        lines += [
            "",
            "The text query and HTAN structured output were used to retrieve relevant medical evidence."
        ]

    return "\n".join(lines)


def interpret_cv_result(cv_result, question="", docs=None, require_citations=True):
    """
    Claude/Sonnet diagnostic-support interpretation.
    Use only in retrieve+answer mode if you intentionally want generated text.

    If docs are provided, the prompt forces citations using PMID/source IDs.
    If docs are not provided, returns local summary instead of making unsupported medical claims.
    """
    if not docs:
        return format_cv_local_summary(cv_result, question=question, mode="retrieve_answer")

    summary_json = _cv_summary_for_llm(cv_result)

    context_lines = []
    for i, d in enumerate(docs[:6], 1):
        source_id = d.get("pmid") or d.get("source") or f"doc{i}"
        context_lines.append(
            f"[{i}] PMID:{source_id}\n"
            f"Title: {d.get('title','')}\n"
            f"Source: {d.get('source','')}\n"
            f"Text: {d.get('text','')[:900]}"
        )

    context = "\n\n".join(context_lines)

    prompt = f"""
You are a careful dermatology diagnostic-support assistant.

You are given:
1. HTAN image segmentation output.
2. Retrieved medical literature snippets.
3. The patient's question.

Important safety rules:
- Do NOT diagnose melanoma or cancer from the image.
- Use "diagnostic-support summary", "screening output", or "professional evaluation recommended".
- Do NOT say "tumor confirmed".
- Every medical claim must cite the retrieved source ID exactly as [PMID:...].
- If the retrieved snippets do not support a claim, say evidence was not found in the retrieved excerpts.
- Recommend dermatologist/clinician evaluation for changing lesions.

HTAN image segmentation JSON:
{summary_json}

Retrieved excerpts:
{context}

Patient question:
{question or "No text question provided."}

Write a concise diagnostic-support summary with:
- Image model findings
- What the retrieved literature says
- Why professional evaluation is recommended
- Clear disclaimer that this is not a diagnosis
"""

    if "call_sonnet" in globals():
        return call_sonnet(
            prompt,
            system="You are a careful dermatology AI assistant. Do not diagnose from an image alone."
        )

    return format_cv_local_summary(cv_result, question=question, mode="retrieve_answer")


try:
    from LangGraph_tool import htan_segmentation_tool
    print("✅ HTAN Cell 23 ready — safe HTAN integration + retrieval augmentation helpers loaded.")
except Exception as e:
    htan_segmentation_tool = None
    print(f"✅ HTAN Cell 23 ready. LangGraph tool import skipped: {e}")


## Stage 17 — Interactive Control Panel
Every toggle and slider controls the actual pipeline. Retrieval evidence is reused for the final answer.


In [ ]:

# =====================================================================
# CELL 24 — INTERACTIVE CONTROL PANEL UI
#           STRICT RETRIEVE-ONLY + CITED DIAGNOSTIC-SUPPORT ANSWER
# =====================================================================
# Requirements:
#   ✅ Cell 17 advanced_retrieve() exists
#   ✅ Cell 23 run_cv_model(), cv_result_to_retrieval_text(), format_cv_local_summary() exist
#   ✅ Optional: Cell 27 run_triage() exists
#   ✅ Optional: call_sonnet() / claude exists for retrieve+answer mode
# =====================================================================

import ipywidgets as w
from IPython.display import display, HTML, clear_output
import time
import numpy as np
import gc
import json
import re
import traceback

clear_output(wait=True)

PANEL_RUNNING = False
PANEL_TRIAGE_PENDING = False

# Close old widgets if cell was rerun
for _old_name in ["panel", "out_area"]:
    if _old_name in globals():
        try:
            globals()[_old_name].close()
        except Exception:
            pass

gc.collect()


# ---------------------------------------------------------------------
# Safe config defaults
# ---------------------------------------------------------------------
USE_QUERY_REWRITER = globals().get("USE_QUERY_REWRITER", True)
USE_HYDE           = globals().get("USE_HYDE", False)
USE_MULTI_QUERY    = globals().get("USE_MULTI_QUERY", True)
USE_DECOMPOSITION  = globals().get("USE_DECOMPOSITION", False)
USE_CROSS_ENCODER  = globals().get("USE_CROSS_ENCODER", True)
USE_COMPRESSION    = globals().get("USE_COMPRESSION", True)
USE_NLI_FILTER     = globals().get("USE_NLI_FILTER", False)
USE_HALLUC_CHECK   = globals().get("USE_HALLUC_CHECK", True)
USE_AUTO_TRIAGE    = globals().get("USE_AUTO_TRIAGE", True)
USE_OOS_GATE       = globals().get("USE_OOS_GATE", True)

SEARCH_K           = globals().get("SEARCH_K", 6)
SEARCH_ALPHA       = globals().get("SEARCH_ALPHA", 0.60)
SEARCH_LAMBDA      = globals().get("SEARCH_LAMBDA", 0.40)
COMPRESSION_THRESHOLD = globals().get("COMPRESSION_THRESHOLD", 0.50)
OOS_THRESHOLD      = globals().get("OOS_THRESHOLD", 0.70)
N_MULTI_QUERIES    = globals().get("N_MULTI_QUERIES", 2)
DISEASE_FILTER     = globals().get("DISEASE_FILTER", None)
CROSS_ENCODER_MODEL = globals().get("CROSS_ENCODER_MODEL", "ncbi/MedCPT-Cross-Encoder")

if "_PANEL_LOADED_CE_MODEL" not in globals() and "cross_encoder" in globals():
    _PANEL_LOADED_CE_MODEL = CROSS_ENCODER_MODEL

# The panel alpha and compression threshold must actually control Cell 17.
PANEL_ALPHA_OVERRIDE_ACTIVE = True
PANEL_COMPRESSION_OVERRIDE_ACTIVE = True

# Save stable defaults for the Reset button.
_PANEL_DEFAULTS = {
    "rewriter": USE_QUERY_REWRITER,
    "hyde": USE_HYDE,
    "multi_q": USE_MULTI_QUERY,
    "decomp": USE_DECOMPOSITION,
    "cross_enc": USE_CROSS_ENCODER,
    "compression": USE_COMPRESSION,
    "nli": USE_NLI_FILTER,
    "halluc": USE_HALLUC_CHECK,
    "auto_triage": USE_AUTO_TRIAGE,
    "oos_gate": USE_OOS_GATE,
    "k": SEARCH_K,
    "alpha": SEARCH_ALPHA,
    "lambda": SEARCH_LAMBDA,
    "comp_t": COMPRESSION_THRESHOLD,
    "oos_t": OOS_THRESHOLD,
    "n_mq": N_MULTI_QUERIES,
    "disease": DISEASE_FILTER,
    "ce_model": CROSS_ENCODER_MODEL,
}

# Permanent fallback for the missing Cell 19 system prompt.
if "CITATION_SYSTEM" not in globals():
    CITATION_SYSTEM = """
You are MediLink, a biomedical evidence-support assistant.
Use only the retrieved evidence supplied in the prompt.
Do not invent medical facts, statistics, treatments, or citations.
Cite important medical claims with the exact supplied source identifier.
Clearly state when evidence is insufficient.
Do not diagnose from an image or from limited symptom information.
Recommend qualified clinical evaluation when appropriate.
""".strip()

# Preserve the original intent-alpha function only once, then make the panel slider effective.
if "get_alpha_for_intent" in globals() and "_MEDILINK_BASE_GET_ALPHA_FOR_INTENT" not in globals():
    _MEDILINK_BASE_GET_ALPHA_FOR_INTENT = globals()["get_alpha_for_intent"]

if "_MEDILINK_BASE_GET_ALPHA_FOR_INTENT" in globals():
    def get_alpha_for_intent(intent):
        if globals().get("PANEL_ALPHA_OVERRIDE_ACTIVE", False):
            return float(globals().get("SEARCH_ALPHA", 0.60))
        return _MEDILINK_BASE_GET_ALPHA_FOR_INTENT(intent)

# Preserve the original compressor only once, then make the panel threshold effective,
# including the symptom path that otherwise hard-codes 0.30.
if "compress_context" in globals() and "_MEDILINK_BASE_COMPRESS_CONTEXT" not in globals():
    _MEDILINK_BASE_COMPRESS_CONTEXT = globals()["compress_context"]

if "_MEDILINK_BASE_COMPRESS_CONTEXT" in globals():
    def compress_context(query, docs, threshold=None, intent=None):
        if globals().get("PANEL_COMPRESSION_OVERRIDE_ACTIVE", False):
            threshold = float(globals().get("COMPRESSION_THRESHOLD", 0.50))
        elif threshold is None:
            threshold = 0.50
        return _MEDILINK_BASE_COMPRESS_CONTEXT(
            query,
            docs,
            threshold=threshold,
            intent=intent,
        )


# ---------------------------------------------------------------------
# CSS
# ---------------------------------------------------------------------
_CSS = """
<style>
.cp-header   { background:#1a1a2e; color:#e0e0ff; padding:14px 20px;
               border-radius:10px 10px 0 0; font-size:17px; font-weight:700;
               letter-spacing:.5px; margin-bottom:0; }
.cp-section  { background:#16213e; color:#a0cfff; padding:6px 14px;
               font-size:12px; font-weight:700; letter-spacing:1.2px;
               text-transform:uppercase; margin:0; }
.cp-card     { background:#0f3460; border-radius:8px; padding:12px 16px;
               margin:6px 0; border-left:3px solid #533483; }
.cp-result   { background:#0d1b2a; color:#c8e6c9; padding:14px 18px;
               border-radius:8px; font-family:monospace; font-size:12.5px;
               white-space:pre-wrap; margin-top:8px; max-height:520px;
               overflow-y:auto; border:1px solid #1e4d2b; }
.cp-err      { background:#2d1b1b; color:#ff8a80; padding:14px;
               border-radius:8px; font-family:monospace; font-size:12px; }
.cp-note     { background:#102a43; color:#bbdefb; padding:10px 14px;
               border-radius:8px; font-size:12px; margin:6px 0; }
.cp-metric   { display:inline-block; background:#1a3a4a; border-radius:6px;
               padding:5px 12px; margin:3px; color:#80cbc4; font-size:12px; }
.cp-badge    { display:inline-block; border-radius:12px; padding:2px 10px;
               font-size:11px; font-weight:700; margin-left:6px; }
.badge-on    { background:#1b5e20; color:#a5d6a7; }
.badge-off   { background:#3e2723; color:#ffab91; }
</style>
"""
display(HTML(_CSS))


# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------
def _html_escape(text):
    return str(text).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")


def _clear_out():
    try:
        out_area.clear_output(wait=True)
    except Exception:
        pass


def _set_panel_buttons(disabled=True, keep_reset=False):
    buttons = [btn_run, btn_triage, btn_bench, btn_oos]
    for btn in buttons:
        try:
            btn.disabled = disabled
        except Exception:
            pass
    try:
        btn_reset.disabled = disabled and not keep_reset
    except Exception:
        pass

def is_smalltalk(message: str) -> bool:
    """Only exact short conversational messages count as small talk."""
    msg = " ".join(str(message).lower().strip().split())
    exact_patterns = {
        "hi", "hello", "hey", "good morning", "good afternoon", "good evening",
        "thanks", "thank you", "ok", "okay", "who are you", "what can you do",
        "how are you", "bye", "goodbye"
    }
    return msg in exact_patterns

def smalltalk_response(message: str) -> str:
    msg = message.lower().strip()
    if any(x in msg for x in ["good morning", "good afternoon", "good evening", "hello", "hi", "hey"]):
        return "Hello. How can I help you today?"
    if any(x in msg for x in ["thank", "thanks"]):
        return "You're welcome."
    if "how are you" in msg:
        return "I'm doing well and ready to help."
    if "what can you do" in msg or "who are you" in msg:
        return "I can help answer questions about melanoma, skin cancer, and multiple myeloma."
    if any(x in msg for x in ["bye", "goodbye"]):
        return "Goodbye. Take care."
    return "How can I help?"


def looks_personal_symptom(message: str) -> bool:
    """Cheap local gate so Haiku triage is called only for personal symptom intake."""
    text = " ".join(str(message).lower().strip().split())
    if not text:
        return False
    patterns = [
        "i have", "i noticed", "i feel", "i am having", "i'm having",
        "my mole", "my skin", "my lesion", "my spot", "my rash",
        "on my body", "it hurts", "it itches", "it is itching", "it bleeds",
        "changed color", "changing color", "getting bigger", "became darker",
        "bone pain", "back pain", "frequent infections", "weight loss",
    ]
    return any(p in text for p in patterns)


def _badge(on):
    return f'<span class="cp-badge {"badge-on" if on else "badge-off"}">{"ON" if on else "OFF"}</span>'


def _flag_summary():
    flags = [
        ("Re-Writer", wi_rewriter.value),
        ("HyDE", wi_hyde.value),
        ("Multi-Query", wi_multi_q.value),
        ("Decomp", wi_decomp.value),
        ("CrossEnc", wi_cross_enc.value),
        ("Compress", wi_compression.value),
        ("NLI", wi_nli.value),
        ("Auto-Triage", wi_auto_triage.value),
        ("OOS", wi_oos_gate.value),
        ("Grounding", wi_halluc.value),
    ]
    return "".join(f'<span class="cp-metric">{n} {_badge(v)}</span>' for n, v in flags)

def _display_docs(docs, elapsed, steps=None, intent=None):
    hdr = (
        f'<div class="cp-header">✅ Retrieved {len(docs)} chunks ({elapsed:.2f}s)'
        + (f' | Intent: <b>{_html_escape(intent)}</b>' if intent else '')
        + '</div>'
        + f'<div style="padding:6px 0;">{_flag_summary()}</div>'
    )

    if steps:
        hdr += (
            '<div style="color:#ffe082;font-size:12px;padding:4px 0;">'
            + "<br>".join(str(s) for s in steps)
            + "</div>"
        )

    display(HTML(hdr))

    for i, doc in enumerate(docs, 1):
        ce = f"CE={float(doc.get('cross_encoder_score', 0)):.3f} | " if "cross_encoder_score" in doc else ""
        cr = f"CR={doc.get('compression_ratio', 1.0):.2f} | " if "compression_ratio" in doc else ""
        rc = f"RC={doc.get('_retrieval_count', 1)}"

        pmid  = _html_escape(doc.get("pmid", "N/A"))
        title = _html_escape(doc.get("title", "Unknown")[:95])
        text  = doc.get("text", "")
        text  = _html_escape(text[:420] + ("…" if len(text) > 420 else ""))
        src   = _html_escape(doc.get("source", "?"))
        ctype = _html_escape(doc.get("chunk_type", "?"))

        display(HTML(f"""
            <div class="cp-card">
              <div style="color:#90caf9;font-size:12px;margin-bottom:4px;">
                <b>[{i}]</b> [{ctype}] [{src}] PMID:{pmid} &nbsp;|&nbsp; {ce}{cr}{rc}
              </div>
              <div style="color:#ffe082;font-size:12.5px;margin-bottom:4px;">{title}</div>
              <div style="color:#cfd8dc;font-size:12px;line-height:1.5;">{text}</div>
            </div>
        """))


def _apply_settings(show_model_status=False):
    global USE_QUERY_REWRITER, USE_HYDE, USE_MULTI_QUERY, N_MULTI_QUERIES
    global USE_DECOMPOSITION, USE_CROSS_ENCODER, USE_COMPRESSION, USE_NLI_FILTER
    global USE_HALLUC_CHECK, USE_AUTO_TRIAGE, USE_OOS_GATE
    global SEARCH_K, SEARCH_ALPHA, SEARCH_LAMBDA
    global COMPRESSION_THRESHOLD, OOS_THRESHOLD, DISEASE_FILTER, CROSS_ENCODER_MODEL
    global PANEL_ALPHA_OVERRIDE_ACTIVE, PANEL_COMPRESSION_OVERRIDE_ACTIVE

    USE_QUERY_REWRITER    = bool(wi_rewriter.value)
    USE_HYDE              = bool(wi_hyde.value)
    USE_MULTI_QUERY       = bool(wi_multi_q.value)
    N_MULTI_QUERIES       = int(wi_n_mq.value)
    USE_DECOMPOSITION     = bool(wi_decomp.value)
    USE_CROSS_ENCODER     = bool(wi_cross_enc.value)
    USE_COMPRESSION       = bool(wi_compression.value)
    USE_NLI_FILTER        = bool(wi_nli.value)
    USE_HALLUC_CHECK      = bool(wi_halluc.value)
    USE_AUTO_TRIAGE       = bool(wi_auto_triage.value)
    USE_OOS_GATE          = bool(wi_oos_gate.value)

    SEARCH_K              = int(wi_k.value)
    SEARCH_ALPHA          = float(wi_alpha.value)
    SEARCH_LAMBDA         = float(wi_lambda.value)
    COMPRESSION_THRESHOLD = float(wi_comp_t.value)
    OOS_THRESHOLD         = float(wi_oos_t.value)
    DISEASE_FILTER        = wi_disease.value
    CROSS_ENCODER_MODEL   = wi_ce_model.value

    PANEL_ALPHA_OVERRIDE_ACTIVE = True
    PANEL_COMPRESSION_OVERRIDE_ACTIVE = True

    if USE_CROSS_ENCODER:
        _ensure_cross_encoder_model(CROSS_ENCODER_MODEL, show_status=show_model_status)

def _uploaded_file_bytes(upload_widget):
    value = upload_widget.value
    if not value:
        return None

    # ipywidgets 7 uses dict; ipywidgets 8 uses tuple/list of dicts.
    if isinstance(value, dict):
        item = next(iter(value.values()))
    elif isinstance(value, (tuple, list)):
        item = value[0]
    else:
        return None

    content = item.get("content") if isinstance(item, dict) else None
    return bytes(content) if content is not None else None


def _get_panel_image_input():
    path = wi_image_path.value.strip()
    if path:
        return path
    return _uploaded_file_bytes(wi_image_upload)


def _display_cv_metrics(cv_result):
    if not cv_result:
        return

    features = cv_result.get("features", {})
    metrics = [
        ("Lesion detected", features.get("tumor_detected")),
        ("Confidence", f"{cv_result.get('confidence', 0):.1%}"),
        ("Area", f"{features.get('tumor_area_percent', 0)}%"),
        ("Lesions", features.get("num_lesions")),
        ("Diameter", f"{features.get('largest_lesion_diameter_mm', 0)} mm"),
        ("Location", features.get("lesion_location")),
        ("Screening priority", features.get("severity_estimate")),
    ]
    html = "".join(
        f'<span class="cp-metric"><b>{_html_escape(name)}</b>: {_html_escape(value)}</span>'
        for name, value in metrics
    )
    display(HTML(f'<div style="padding:6px 0;">{html}</div>'))


def _make_augmented_query(question, cv_result):
    """
    Combines user text + HTAN structured output for retrieval.
    This is the key: the image model output augments retrieval, not free diagnosis.
    """
    if "cv_result_to_retrieval_text" in globals():
        cv_text = cv_result_to_retrieval_text(cv_result)
    else:
        f = cv_result.get("features", {}) if cv_result else {}
        cv_text = (
            "HTAN image segmentation structured output: "
            f"lesion_detected={f.get('tumor_detected', False)}; "
            f"largest_lesion_diameter_mm={f.get('largest_lesion_diameter_mm', 0)}; "
            f"screening_priority={f.get('severity_estimate', 'unknown')}."
        )

    if question:
        return f"{question}\n\n{cv_text}"
    return cv_text


def _build_cited_multimodal_prompt(original_question, augmented_query, cv_result, docs, intent):
    """
    Build final cited diagnostic-support prompt using the exact retrieved docs.
    This avoids calling advanced_rag_answer() again and losing the HTAN augmented retrieval context.
    """
    if "format_cv_local_summary" in globals():
        cv_summary = format_cv_local_summary(cv_result, question=original_question, mode="retrieve_answer")
    else:
        cv_summary = json.dumps(cv_result.get("features", {}), indent=2, default=str)

    lines = [
        "You are a careful medical AI assistant providing a diagnostic-support summary.",
        "",
        "Strict rules:",
        "- Do NOT diagnose melanoma or cancer from the image.",
        "- Do NOT say tumor/cancer is confirmed.",
        "- Use terms like 'lesion detected', 'screening output', and 'professional evaluation recommended'.",
        "- Use the HTAN output only as image segmentation/screening context.",
        "- Use ONLY the retrieved excerpts for medical claims.",
        "- Cite every medical claim using the source ID exactly like [PMID:...].",
        "- If evidence is not found in the retrieved excerpts, say so.",
        "- Recommend clinician/dermatology review for changing lesions.",
        "",
        f"Intent: {intent}",
        "",
        "HTAN STRUCTURED OUTPUT:",
        cv_summary,
        "",
        "AUGMENTED RETRIEVAL QUERY:",
        augmented_query,
        "",
        "RETRIEVED EXCERPTS:"
    ]

    if not docs:
        lines.append("No retrieved excerpts were available.")
    else:
        for i, d in enumerate(docs, 1):
            source_id = d.get("pmid") or d.get("source") or f"doc{i}"
            lines.append(
                f"\n[{i}] PMID:{source_id} [{d.get('chunk_type','?').upper()}]\n"
                f"Title: {d.get('title','')}\n"
                f"Source: {d.get('source','')}\n"
                f"Text: {d.get('text','')}"
            )

    lines += [
        "",
        f"PATIENT QUESTION: {original_question}",
        "",
        "Write the final answer with these sections:",
        "1. Image analysis summary",
        "2. Evidence-based interpretation with citations",
        "3. What the user should do next",
        "4. Safety disclaimer",
        "",
        "ANSWER:"
    ]

    return "\n".join(lines)


def _call_cited_multimodal_answer(original_question, augmented_query, cv_result, docs, intent):
    prompt = _build_cited_multimodal_prompt(
        original_question=original_question,
        augmented_query=augmented_query,
        cv_result=cv_result,
        docs=docs,
        intent=intent,
    )

    if "call_sonnet" in globals():
        answer = call_sonnet(
            prompt,
            system=(
                "You are a careful dermatology diagnostic-support assistant. "
                "Use only supplied evidence and do not diagnose from an image alone."
            ),
            history=globals().get("conversation_history") or None,
        )
        return _postprocess_answer(
            answer,
            docs,
            original_question=original_question,
            add_disclaimer=True,
        )

    if "call_haiku" in globals():
        answer = call_haiku(prompt, max_tokens=900)
        return _postprocess_answer(
            answer,
            docs,
            original_question=original_question,
            add_disclaimer=True,
        )

    return (
        "Claude is not configured, so no generated diagnostic-support answer was produced.\n\n"
        "Retrieved evidence is shown below."
    )

# ---------------------------------------------------------------------
# Pipeline execution helpers
# ---------------------------------------------------------------------
def _finish_panel_run():
    global PANEL_RUNNING, PANEL_TRIAGE_PENDING
    PANEL_RUNNING = False
    PANEL_TRIAGE_PENDING = False
    _set_panel_buttons(False)


def _set_triage_pending():
    global PANEL_RUNNING, PANEL_TRIAGE_PENDING
    PANEL_RUNNING = False
    PANEL_TRIAGE_PENDING = True
    _set_panel_buttons(True, keep_reset=True)


def _ensure_cross_encoder_model(model_name, show_status=False):
    """Load/reuse the cross-encoder selected in the panel."""
    global cross_encoder, CROSS_ENCODER_MODEL, _PANEL_LOADED_CE_MODEL

    if show_status:
        with out_area:
            display(HTML(
                f'<div class="cp-note">⏳ Preparing cross-encoder: {_html_escape(model_name)}</div>'
            ))

    cross_encoder = ensure_cross_encoder(model_name)
    CROSS_ENCODER_MODEL = model_name
    _PANEL_LOADED_CE_MODEL = model_name


def _prepare_retrieval_query(base_query, intent, steps=None, cv_result=None):
    """Apply the panel rewriter exactly once, then optionally append HTAN output."""
    query = (base_query or "").strip()
    rewritten = query

    if USE_QUERY_REWRITER and query and "rewrite_query" in globals():
        try:
            rewritten = rewrite_query(query, verbose=False, intent=intent)
        except TypeError:
            rewritten = rewrite_query(query, verbose=False)
        if steps is not None and rewritten != query:
            steps.append(f"✏️ Re-written: {_html_escape(rewritten)}")

    if cv_result is not None:
        rewritten = _make_augmented_query(rewritten, cv_result)
        if steps is not None:
            steps.append("🧩 HTAN structured output appended to the retrieval query.")

    return rewritten


def _run_advanced_retrieve_panel(query, intent, disease_override=None):
    """Run advanced_retrieve once using the exact active panel settings."""
    global USE_QUERY_REWRITER

    saved_rewriter = USE_QUERY_REWRITER
    USE_QUERY_REWRITER = False
    try:
        return advanced_retrieve(
            query,
            k=SEARCH_K,
            use_hyde=USE_HYDE,
            use_multi_q=USE_MULTI_QUERY,
            use_cross_enc=USE_CROSS_ENCODER,
            use_compress=USE_COMPRESSION,
            use_decomp=USE_DECOMPOSITION,
            use_nli_filter=USE_NLI_FILTER,
            disease=(
                DISEASE_FILTER
                if disease_override is None
                else disease_override
            ),
            verbose=False,
            intent=intent,
            alpha=SEARCH_ALPHA if PANEL_ALPHA_OVERRIDE_ACTIVE else None,
        )
    finally:
        USE_QUERY_REWRITER = saved_rewriter


def _run_panel_benchmark(benchmark):
    """Run benchmark using the exact active panel configuration."""
    results = []
    for item in benchmark:
        question = item["question"]
        ground_truth = item["ground_truth"]
        intent = classify_intent(question) if "classify_intent" in globals() else "general"
        retrieval_query = _prepare_retrieval_query(question, intent, steps=None)
        disease = DISEASE_FILTER if DISEASE_FILTER is not None else item.get("disease")
        docs = _run_advanced_retrieve_panel(
            retrieval_query,
            intent,
            disease_override=disease,
        )
        contexts = [d.get("text", "") for d in docs if d.get("text")]

        relevance = context_relevance(question, contexts) if contexts else 0.0
        coverage = context_coverage(ground_truth, contexts) if contexts else 0.0
        faithful = answer_faithfulness(ground_truth, contexts) if contexts else 0.0
        diversity = chunk_diversity(contexts) if contexts else 0.0
        avg = float(np.mean([relevance, coverage, faithful]))

        results.append({
            "disease": item.get("disease", disease or "unknown"),
            "question": question[:65],
            "relevance": round(float(relevance), 3),
            "coverage": round(float(coverage), 3),
            "faithful": round(float(faithful), 3),
            "diversity": round(float(diversity), 3),
            "avg": round(avg, 3),
        })
    return results


def _source_id(doc, index):
    return str(doc.get("pmid") or doc.get("source") or f"doc{index}")


def _build_cited_text_prompt(original_question, retrieval_query, docs, intent):
    lines = [
        "You are a biomedical evidence-support assistant.",
        "Use ONLY the retrieved excerpts for medical claims.",
        "Cite every important medical claim using the exact supplied source ID.",
        "Do not invent evidence, statistics, diagnoses, or citations.",
        "State clearly when the evidence is insufficient.",
        "This is educational information, not a diagnosis.",
        "",
        f"Intent: {intent}",
        f"Retrieval query: {retrieval_query}",
        "",
        "RETRIEVED EXCERPTS:",
    ]

    if not docs:
        lines.append("No retrieved excerpts were available.")
    else:
        for i, doc in enumerate(docs, 1):
            sid = _source_id(doc, i)
            lines.append(
                f"\n[{i}] PMID:{sid} [{str(doc.get('chunk_type', '?')).upper()}]\n"
                f"Title: {doc.get('title', '')}\n"
                f"Source: {doc.get('source', '')}\n"
                f"Text: {doc.get('text', '')}"
            )

    lines += [
        "",
        f"USER QUESTION: {original_question}",
        "",
        "Write a clear answer with inline citations such as [PMID:12345678].",
        "ANSWER:",
    ]
    return "\n".join(lines)


def _postprocess_answer(answer, docs, original_question=None, add_disclaimer=True):
    """Reuse notebook verification without repeating retrieval."""
    if "verify_citations" in globals():
        try:
            check = verify_citations(answer, docs)
            bad = check.get("hallucinated_pmids", []) if isinstance(check, dict) else []
            if bad:
                pattern = r"PMID:(?:" + "|".join(re.escape(str(x)) for x in bad) + r")"
                answer = re.sub(pattern, "PMID:[unverified]", answer)
        except Exception:
            pass

    if USE_HALLUC_CHECK and "verify_answer_grounding" in globals():
        try:
            answer = verify_answer_grounding(answer, docs)
        except Exception as error:
            answer += f"\n\n[Grounding check unavailable: {error}]"

    if add_disclaimer:
        disclaimer = globals().get(
            "MEDICAL_DISCLAIMER",
            "\n\n⚠️ Educational support only; this is not a medical diagnosis."
        )
        if disclaimer and disclaimer not in answer:
            answer += disclaimer

    if original_question and "update_history" in globals():
        try:
            update_history("user", original_question)
            update_history("assistant", answer)
        except Exception:
            pass

    return answer


def _call_cited_text_answer(original_question, retrieval_query, docs, intent):
    if not docs:
        return "No sufficiently relevant evidence was retrieved."
    if "call_sonnet" not in globals():
        return "Claude Sonnet is not configured, so no generated answer was produced."

    prompt = _build_cited_text_prompt(
        original_question=original_question,
        retrieval_query=retrieval_query,
        docs=docs,
        intent=intent,
    )
    history = globals().get("conversation_history")
    answer = call_sonnet(
        prompt,
        system=CITATION_SYSTEM,
        history=history if history else None,
    )
    return _postprocess_answer(
        answer,
        docs,
        original_question=original_question,
        add_disclaimer=True,
    )


def _run_oos_gate(query, steps, skip_reason=None):
    if not USE_OOS_GATE:
        steps.append("🚨 OOS gate OFF by panel choice.")
        return True, None
    if skip_reason:
        steps.append(f"🚨 OOS skipped ✅ — {skip_reason}")
        return True, None
    if "is_in_scope" not in globals():
        steps.append("🚨 OOS unavailable — is_in_scope() is not loaded.")
        return True, None

    in_scope, score = is_in_scope(query, threshold=OOS_THRESHOLD)
    steps.append(f"🚨 OOS score={score:.3f} {'✅ IN SCOPE' if in_scope else '❌ BLOCKED'}")
    return bool(in_scope), float(score)

# ---------------------------------------------------------------------
# Widgets
# ---------------------------------------------------------------------
_lbl_query = w.HTML('<div class="cp-section">📝 Query</div>')

wi_question = w.Textarea(
    value="my skin has a dark spot that changed color",
    layout=w.Layout(width="100%", height="72px"),
    style={"description_width": "0"}
)

_lbl_image = w.HTML('<div class="cp-section">🔬 HTAN Image Analysis</div>')

wi_image_upload = w.FileUpload(
    accept=".jpg,.jpeg,.png",
    multiple=False,
    description="Upload image",
    layout=w.Layout(width="220px")
)

wi_image_path = w.Text(
    value="",
    placeholder="/full/path/to/skin_image.jpg",
    description="Image path:",
    style={"description_width": "90px"},
    layout=w.Layout(width="520px"),
)

wi_image_type = w.Dropdown(
    options=[
        ("Skin lesion / dermoscopy", "skin_lesion"),
        ("Other medical image", "medical_image"),
        ("Medical document", "medical_document"),
        ("Non-medical image", "non_medical"),
        ("Blurry image", "blurry"),
    ],
    value="skin_lesion",
    description="Image type:",
    style={"description_width": "90px"},
    layout=w.Layout(width="300px"),
)

wi_pixel_spacing = w.FloatText(
    value=globals().get("HTAN_PIXEL_SPACING_MM", 0.1),
    description="mm/pixel:",
    style={"description_width": "80px"},
    layout=w.Layout(width="180px"),
)

row_image_input = w.HBox([wi_image_upload, wi_image_path], layout=w.Layout())
row_image_settings = w.HBox([wi_image_type, wi_pixel_spacing], layout=w.Layout())

wi_disease = w.Dropdown(
    options=[
        ("All diseases (no filter)", None),
        ("Multiple myeloma", "multiple myeloma"),
        ("Melanoma", "melanoma"),
        ("Skin cancer", "skin cancer")
    ],
    value=DISEASE_FILTER,
    description="Disease filter:",
    style={"description_width": "130px"},
    layout=w.Layout(width="380px"),
)

wi_k = w.IntSlider(
    value=SEARCH_K,
    min=1,
    max=20,
    step=1,
    description="Top-K chunks:",
    style={"description_width": "120px"},
    layout=w.Layout(width="400px"),
)

row_query_settings = w.HBox([wi_disease, wi_k], layout=w.Layout())

_lbl_pipe = w.HTML('<div class="cp-section">⚙️ Pipeline Toggles</div>')


def _toggle(label, value, icon=""):
    return w.Checkbox(
        value=value,
        description=f"{icon}  {label}",
        layout=w.Layout(width="220px"),
        style={"description_width": "0"},
        indent=False,
    )


wi_rewriter    = _toggle("Query Re-Writer", USE_QUERY_REWRITER, "✏️")
wi_hyde        = _toggle("HyDE Expansion",  USE_HYDE,           "💡")
wi_multi_q     = _toggle("Multi-Query",     USE_MULTI_QUERY,    "🔀")
wi_decomp      = _toggle("Decomposition",   USE_DECOMPOSITION,  "✂️")
wi_cross_enc   = _toggle("Cross-Encoder",   USE_CROSS_ENCODER,  "🎯")
wi_compression = _toggle("Compression",     USE_COMPRESSION,    "🗜️")
wi_nli         = _toggle("NLI Filter",      USE_NLI_FILTER,     "🧪")
wi_auto_triage = _toggle("Auto-Triage",     USE_AUTO_TRIAGE,    "🩺")
wi_oos_gate    = _toggle("OOS Gate",        USE_OOS_GATE,       "🚨")
wi_halluc      = _toggle("Grounding Check", USE_HALLUC_CHECK,   "🛡️")

row_toggles = w.GridBox(
    [
        wi_rewriter, wi_hyde, wi_multi_q,
        wi_decomp, wi_cross_enc, wi_compression,
        wi_nli, wi_auto_triage, wi_oos_gate,
        wi_halluc,
    ],
    layout=w.Layout(grid_template_columns="repeat(3, 230px)", grid_gap="4px 20px")
)

_lbl_search = w.HTML('<div class="cp-section">🔍 Search Parameters</div>')


def _slider(desc, val, lo, hi, step, fmt=".2f"):
    return w.FloatSlider(
        value=val,
        min=lo,
        max=hi,
        step=step,
        description=desc,
        readout_format=fmt,
        style={"description_width": "160px"},
        layout=w.Layout(width="440px"),
    )


wi_alpha  = _slider("Hybrid alpha (semantic):", SEARCH_ALPHA,          0.0, 1.0, 0.05)
wi_lambda = _slider("MMR lambda (relevance):",  SEARCH_LAMBDA,         0.0, 1.0, 0.05)
wi_comp_t = _slider("Compression threshold:",   COMPRESSION_THRESHOLD, 0.1, 0.9, 0.05)
wi_oos_t  = _slider("OOS threshold:",           OOS_THRESHOLD,         0.1, 0.99, 0.05)

wi_n_mq = w.IntSlider(
    value=N_MULTI_QUERIES,
    min=1,
    max=8,
    step=1,
    description="Num multi-queries:",
    style={"description_width": "160px"},
    layout=w.Layout(width="440px"),
)

row_params1 = w.HBox([wi_alpha,  wi_lambda], layout=w.Layout())
row_params2 = w.HBox([wi_comp_t, wi_oos_t],  layout=w.Layout())
row_params3 = w.HBox([wi_n_mq])

_lbl_model = w.HTML('<div class="cp-section">🤖 Model & Mode</div>')

ce_options = [
    "ncbi/MedCPT-Cross-Encoder",
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    "cross-encoder/ms-marco-MiniLM-L-12-v2",
    "cross-encoder/ms-marco-TinyBERT-L-2-v2",
]

if CROSS_ENCODER_MODEL not in ce_options:
    ce_options.insert(0, CROSS_ENCODER_MODEL)

wi_ce_model = w.Dropdown(
    options=ce_options,
    value=CROSS_ENCODER_MODEL,
    description="Cross-Encoder model:",
    style={"description_width": "160px"},
    layout=w.Layout(width="500px"),
)

wi_run_mode = w.RadioButtons(
    options=[
        "retrieve only",
        "retrieve + diagnostic-support answer (Claude API)"
    ],
    value="retrieve only",
    description="Run mode:",
    style={"description_width": "100px"},
)

row_model = w.HBox([wi_ce_model], layout=w.Layout(margin="0 0 6px 0"))

btn_run    = w.Button(description="▶  Run Pipeline",   button_style="success", layout=w.Layout(width="200px", height="40px"))
btn_triage = w.Button(description="🩺  Triage",         button_style="info",    layout=w.Layout(width="160px", height="40px"))
btn_bench  = w.Button(description="📊  Run Benchmark", button_style="warning", layout=w.Layout(width="200px", height="40px"))
btn_oos    = w.Button(description="🚨  OOS Test",      button_style="danger",  layout=w.Layout(width="160px", height="40px"))
btn_reset  = w.Button(description="↺  Reset Defaults", button_style="",        layout=w.Layout(width="160px", height="40px"))

out_area = w.Output(layout=w.Layout(margin="8px 0 0 0"))


# ---------------------------------------------------------------------
# Triage handling
# ---------------------------------------------------------------------
def _run_haiku_triage_then_rag(question):
    """Show triage only when selected and complete the same one-pass RAG path."""
    if not USE_AUTO_TRIAGE or "run_triage" not in globals():
        return False

    def _triage_done(clinical_query):
        global PANEL_RUNNING, PANEL_TRIAGE_PENDING
        PANEL_TRIAGE_PENDING = False
        PANEL_RUNNING = True
        _set_panel_buttons(True, keep_reset=True)
        try:
            _run_text_retrieval_pipeline(
                original_question=question,
                retrieval_base=clinical_query,
                forced_intent="symptoms",
                initial_steps=[
                    "🩺 Haiku converted the personal symptom description into a clinical retrieval query."
                ],
            )
        finally:
            _finish_panel_run()

    with out_area:
        display(HTML('<div class="cp-note">🩺 Asking Haiku if triage is needed…</div>'))

    triggered = run_triage(question, on_result=_triage_done)
    if triggered:
        _set_triage_pending()
    return bool(triggered)

def _run_text_retrieval_pipeline(
    original_question,
    retrieval_base=None,
    forced_intent=None,
    initial_steps=None,
):
    """One retrieval pass; answer mode reuses the same retrieved documents."""
    steps = list(initial_steps or [])
    base_query = (retrieval_base or original_question or "").strip()

    intent = forced_intent or (
        classify_intent(base_query) if "classify_intent" in globals() else "general"
    )
    retrieval_query = _prepare_retrieval_query(base_query, intent, steps=steps)

    in_scope, score = _run_oos_gate(
        base_query or retrieval_query,
        steps,
        skip_reason="triage confirmed an in-scope symptom case" if forced_intent == "symptoms" else None,
    )
    if not in_scope:
        _clear_out()
        with out_area:
            display(HTML(
                f'<div class="cp-err">🚨 Out-of-scope query '
                f'(score={score:.3f}, threshold={OOS_THRESHOLD:.2f}).</div>'
            ))
        return

    _clear_out()
    with out_area:
        display(HTML('<div class="cp-header">⏳ Running text RAG pipeline…</div>'))

    t0 = time.time()
    docs = _run_advanced_retrieve_panel(retrieval_query, intent)
    elapsed = time.time() - t0

    _clear_out()
    with out_area:
        _display_docs(docs, elapsed, steps, intent=intent)

        if wi_run_mode.value.startswith("retrieve +"):
            display(HTML('<div class="cp-note">🤖 Calling Claude once using the already retrieved evidence…</div>'))
            answer = _call_cited_text_answer(
                original_question=original_question,
                retrieval_query=retrieval_query,
                docs=docs,
                intent=intent,
            )
            display(HTML(f'<div class="cp-result">{_html_escape(answer)}</div>'))
        else:
            display(HTML(
                '<div class="cp-note">Retrieve-only mode: no final Claude answer was called.</div>'
            ))


# ---------------------------------------------------------------------
# Image + RAG handling
# ---------------------------------------------------------------------
def _run_image_retrieval_pipeline(question, image_input, clinical_query=None):
    """HTAN + one advanced retrieval pass + optional one final Claude answer."""
    globals()["HTAN_PIXEL_SPACING_MM"] = wi_pixel_spacing.value

    with out_area:
        display(HTML('<div class="cp-header">🔬 Running HTAN image analysis…</div>'))

    cv_result = run_cv_model(
        image_input,
        image_type=wi_image_type.value,
        pixel_spacing_mm=wi_pixel_spacing.value,
    )

    text_is_smalltalk_with_image = bool(question and is_smalltalk(question) and not clinical_query)

    if text_is_smalltalk_with_image:
        retrieval_base = ""
        question_for_answer = "The user provided a skin-lesion image for screening support."
    else:
        retrieval_base = (clinical_query or question or "").strip()
        question_for_answer = question or "Please explain the image screening output."

    intent = (
        "symptoms"
        if clinical_query or not retrieval_base
        else classify_intent(retrieval_base) if "classify_intent" in globals() else "symptoms"
    )

    steps = ["🔬 HTAN image segmentation completed."]
    if clinical_query:
        steps.append("🩺 Triage completed; its clinical query is used for retrieval.")
    if text_is_smalltalk_with_image:
        steps.append("💬 Exact small talk was ignored; HTAN output drives medical retrieval.")

    retrieval_query = _prepare_retrieval_query(
        retrieval_base,
        intent,
        steps=steps,
        cv_result=cv_result,
    )

    skip_reason = None
    if clinical_query:
        skip_reason = "triage confirmed an in-scope symptom case"
    elif text_is_smalltalk_with_image:
        skip_reason = "the medical signal comes from the image, not the greeting"
    elif not retrieval_base:
        skip_reason = "image-only analysis uses HTAN output"

    in_scope, score = _run_oos_gate(
        retrieval_base or retrieval_query,
        steps,
        skip_reason=skip_reason,
    )

    if not in_scope:
        _clear_out()
        with out_area:
            display(HTML('<div class="cp-header">🔬 HTAN Complete | RAG blocked by OOS gate</div>'))
            _display_cv_metrics(cv_result)
            cv_text = (
                format_cv_local_summary(cv_result, question=question_for_answer, mode="retrieve_only")
                if "format_cv_local_summary" in globals()
                else str(cv_result)
            )
            display(HTML(f'<div class="cp-result">{_html_escape(cv_text)}</div>'))
            display(HTML(
                f'<div class="cp-err">🚨 OOS blocked the text query. '
                f'Score={score:.3f}, threshold={OOS_THRESHOLD:.2f}.</div>'
            ))
        return

    t0 = time.time()
    docs = _run_advanced_retrieve_panel(retrieval_query, intent)
    elapsed = time.time() - t0

    _clear_out()
    with out_area:
        answer_mode = wi_run_mode.value.startswith("retrieve +")
        mode = "retrieve_answer" if answer_mode else "retrieve_only"
        display(HTML(
            f'<div class="cp-header">🔬 HTAN Complete | Path: '
            f'medical_image_medical_text_htan_{mode}</div>'
        ))
        _display_cv_metrics(cv_result)

        if clinical_query:
            display(HTML(
                f'<div class="cp-note">🩺 Clinical retrieval query: '
                f'{_html_escape(clinical_query)}</div>'
            ))

        if answer_mode:
            display(HTML(
                '<div class="cp-note">🤖 Calling Claude once using the HTAN output and already retrieved evidence…</div>'
            ))
            cv_intro = (
                format_cv_local_summary(cv_result, question=question_for_answer, mode="retrieve_answer")
                if "format_cv_local_summary" in globals()
                else json.dumps(cv_result.get("features", {}), indent=2, default=str)
            )
            final_answer = _call_cited_multimodal_answer(
                original_question=question_for_answer,
                augmented_query=retrieval_query,
                cv_result=cv_result,
                docs=docs,
                intent=intent,
            )
            display(HTML(
                f'<div class="cp-result">{_html_escape(cv_intro + chr(10) + chr(10) + final_answer)}</div>'
            ))
        else:
            cv_text = (
                format_cv_local_summary(cv_result, question=question_for_answer, mode="retrieve_only")
                if "format_cv_local_summary" in globals()
                else json.dumps(cv_result.get("features", {}), indent=2, default=str)
            )
            display(HTML(f'<div class="cp-result">{_html_escape(cv_text)}</div>'))
            display(HTML(
                '<div class="cp-note">Retrieve-only mode: HTAN output + retrieved chunks; no final Claude answer.</div>'
            ))

        _display_docs(docs, elapsed, steps, intent=intent)

def _run_haiku_triage_then_image(question, image_input):
    """Personal image symptoms can be triaged before HTAN; general image questions skip triage."""
    if not USE_AUTO_TRIAGE or not question or "run_triage" not in globals():
        return False

    def _triage_done(clinical_query):
        global PANEL_RUNNING, PANEL_TRIAGE_PENDING
        PANEL_TRIAGE_PENDING = False
        PANEL_RUNNING = True
        _set_panel_buttons(True, keep_reset=True)
        try:
            _clear_out()
            with out_area:
                display(HTML(
                    f'<div class="cp-header">🩺 Haiku Triage Complete</div>'
                    f'<div class="cp-note">Clinical query: {_html_escape(clinical_query)}</div>'
                ))
            _run_image_retrieval_pipeline(
                question,
                image_input,
                clinical_query=clinical_query,
            )
        finally:
            _finish_panel_run()

    with out_area:
        display(HTML('<div class="cp-note">🩺 Asking Haiku if image + symptom triage is needed…</div>'))

    triggered = run_triage(question, on_result=_triage_done)
    if triggered:
        _set_triage_pending()
    return bool(triggered)

# ---------------------------------------------------------------------
# Button handlers
# ---------------------------------------------------------------------
def _on_run(b):
    global PANEL_RUNNING

    if PANEL_RUNNING or PANEL_TRIAGE_PENDING:
        return

    PANEL_RUNNING = True
    _set_panel_buttons(True, keep_reset=True)
    defer_finish = False

    try:
        _apply_settings(show_model_status=True)
        question = wi_question.value.strip()
        image_input = _get_panel_image_input()

        if not question and image_input is None:
            _clear_out()
            with out_area:
                display(HTML('<div class="cp-err">⚠️ Enter a question or add an image.</div>'))
            return

        if image_input is not None:
            _clear_out()
            should_triage = (
                USE_AUTO_TRIAGE
                and question
                and not is_smalltalk(question)
                and looks_personal_symptom(question)
            )
            if should_triage and _run_haiku_triage_then_image(question, image_input):
                defer_finish = True
                return

            _run_image_retrieval_pipeline(question, image_input)
            return

        if is_smalltalk(question):
            _clear_out()
            with out_area:
                display(HTML(
                    f'<div class="cp-header">💬 Chat</div>'
                    f'<div class="cp-result">{_html_escape(smalltalk_response(question))}</div>'
                ))
            return

        should_triage = USE_AUTO_TRIAGE and looks_personal_symptom(question)
        if should_triage and _run_haiku_triage_then_rag(question):
            defer_finish = True
            return

        _run_text_retrieval_pipeline(original_question=question)

    except Exception as error:
        _clear_out()
        with out_area:
            display(HTML(
                f'<div class="cp-err">❌ {type(error).__name__}: {_html_escape(str(error))}</div>'
            ))
        traceback.print_exc()

    finally:
        if not defer_finish:
            _finish_panel_run()

def _on_triage(b):
    global PANEL_RUNNING

    if PANEL_RUNNING or PANEL_TRIAGE_PENDING:
        return

    PANEL_RUNNING = True
    _set_panel_buttons(True, keep_reset=True)
    defer_finish = False

    try:
        _apply_settings(show_model_status=True)
        question = wi_question.value.strip()
        _clear_out()

        if not question:
            with out_area:
                display(HTML('<div class="cp-err">⚠️ Please enter a question.</div>'))
            return

        if "run_triage" not in globals():
            with out_area:
                display(HTML('<div class="cp-err">run_triage() is not loaded.</div>'))
            return

        with out_area:
            display(HTML('<div class="cp-header">🩺 Manual Haiku Triage</div>'))

        # Manual button always asks triage, independent of Auto-Triage checkbox.
        saved_auto = USE_AUTO_TRIAGE
        globals()["USE_AUTO_TRIAGE"] = True
        try:
            triggered = _run_haiku_triage_then_rag(question)
        finally:
            globals()["USE_AUTO_TRIAGE"] = saved_auto

        if triggered:
            defer_finish = True
        else:
            with out_area:
                display(HTML('<div class="cp-note">✅ No triage needed. Use Run Pipeline.</div>'))

    except Exception as error:
        _clear_out()
        with out_area:
            display(HTML(
                f'<div class="cp-err">❌ Triage error: {_html_escape(str(error))}</div>'
            ))
        traceback.print_exc()

    finally:
        if not defer_finish:
            _finish_panel_run()

def _on_bench(b):
    global PANEL_RUNNING

    if PANEL_RUNNING:
        return

    PANEL_RUNNING = True
    _set_panel_buttons(True)

    try:
        _apply_settings(show_model_status=True)
        _clear_out()

        with out_area:
            display(HTML('<div class="cp-header">⏳ Running Benchmark… please wait</div>'))

        t0 = time.time()
        results = _run_panel_benchmark(BENCHMARK_QUESTIONS)
        elapsed = time.time() - t0

        rows_html = ""
        diseases = ["multiple myeloma", "melanoma", "skin cancer"]

        for disease in diseases:
            group = [r for r in results if r["disease"] == disease]

            for r in group:
                rows_html += (
                    f'<tr><td style="color:#90caf9">{_html_escape(r["disease"])}</td>'
                    f'<td style="color:#e0e0e0">{_html_escape(r["question"][:55])}</td>'
                    f'<td>{r["relevance"]:.3f}</td><td>{r["coverage"]:.3f}</td>'
                    f'<td>{r["faithful"]:.3f}</td><td>{r["diversity"]:.3f}</td>'
                    f'<td style="color:#ffe082;font-weight:700">{r["avg"]:.3f}</td></tr>'
                )

            if group:
                d_avg = {
                    m: np.mean([r[m] for r in group])
                    for m in ["relevance", "coverage", "faithful", "diversity", "avg"]
                }

                rows_html += (
                    f'<tr style="background:#1a3a2a">'
                    f'<td colspan="2" style="color:#a5d6a7;font-weight:700">→ {_html_escape(disease)} AVG</td>'
                    f'<td>{d_avg["relevance"]:.3f}</td><td>{d_avg["coverage"]:.3f}</td>'
                    f'<td>{d_avg["faithful"]:.3f}</td><td>{d_avg["diversity"]:.3f}</td>'
                    f'<td style="color:#69f0ae;font-weight:700">{d_avg["avg"]:.3f}</td></tr>'
                )

        overall = {
            m: np.mean([r[m] for r in results])
            for m in ["relevance", "coverage", "faithful", "diversity", "avg"]
        }

        grade = "🟢 Excellent" if overall["avg"] > 0.75 else "🟡 Good" if overall["avg"] > 0.60 else "🔴 Needs work"

        _clear_out()

        with out_area:
            display(HTML(f"""
            <div class="cp-header">📊 Benchmark done — {elapsed:.1f}s | Score: {overall['avg']:.3f} {grade}</div>
            <table style="width:100%;border-collapse:collapse;font-size:12px;background:#0d1b2a;color:#cfd8dc;">
              <thead><tr style="background:#1a3a5a;color:#80cbc4">
                <th style="padding:6px;text-align:left">Disease</th>
                <th style="text-align:left">Question</th>
                <th>Relev</th><th>Cover</th><th>Faith</th><th>Divers</th>
                <th style="color:#ffe082">AVG</th>
              </tr></thead>
              <tbody>{rows_html}</tbody>
            </table>
            """))

    except Exception as e:
        _clear_out()
        with out_area:
            display(HTML(f'<div class="cp-err">❌ Benchmark error: {_html_escape(str(e))}</div>'))
        raise e

    finally:
        _finish_panel_run()


def _on_oos(b):
    global PANEL_RUNNING

    if PANEL_RUNNING:
        return

    PANEL_RUNNING = True
    _set_panel_buttons(True)

    try:
        _apply_settings(show_model_status=True)
        _clear_out()

        with out_area:
            display(HTML('<div class="cp-header">⏳ Running OOS Test… please wait</div>'))

        oos_rows, in_rows = "", ""
        oos_scores_lst, in_scores_lst = [], []

        for q in OUT_OF_SCOPE_QUERIES:
            in_s, score = is_in_scope(q, threshold=OOS_THRESHOLD)
            oos_scores_lst.append(score)
            icon  = "✅ BLOCKED" if not in_s else "⚠️ LEAKED"
            color = "#a5d6a7" if not in_s else "#ff8a80"
            oos_rows += f'<tr><td style="color:{color}">{icon}</td><td>{score:.3f}</td><td>{_html_escape(q[:65])}</td></tr>'

        for q in IN_SCOPE_QUERIES:
            in_s, score = is_in_scope(q, threshold=OOS_THRESHOLD)
            in_scores_lst.append(score)
            icon  = "✅ PASSED" if in_s else "❌ MISSED"
            color = "#a5d6a7" if in_s else "#ff8a80"
            in_rows += f'<tr><td style="color:{color}">{icon}</td><td>{score:.3f}</td><td>{_html_escape(q[:65])}</td></tr>'

        avg_oos    = np.mean(oos_scores_lst)
        avg_in     = np.mean(in_scores_lst)
        block_rate = sum(1 for s in oos_scores_lst if s < OOS_THRESHOLD) / len(oos_scores_lst) * 100
        pass_rate  = sum(1 for s in in_scores_lst  if s >= OOS_THRESHOLD) / len(in_scores_lst)  * 100

        tbl = "width:100%;border-collapse:collapse;font-size:12px;background:#0d1b2a;color:#cfd8dc"
        th  = "background:#2a1a3a;color:#ce93d8;padding:5px 10px;text-align:left"

        _clear_out()

        with out_area:
            display(HTML(f"""
            <div class="cp-header">🚨 OOS Test Complete</div>
            <div style="padding:6px 0;font-size:12px;">
              <span class="cp-metric">Avg OOS {avg_oos:.3f}</span>
              <span class="cp-metric">Avg IS {avg_in:.3f}</span>
              <span class="cp-metric">Block {block_rate:.0f}%</span>
              <span class="cp-metric">Pass {pass_rate:.0f}%</span>
            </div>
            <b style="color:#ce93d8">Out-of-scope</b>
            <table style="{tbl}"><thead><tr>
              <th style="{th}">Status</th><th style="{th}">Score</th><th style="{th}">Query</th>
            </tr></thead><tbody>{oos_rows}</tbody></table>
            <br><b style="color:#80cbc4">In-scope</b>
            <table style="{tbl}"><thead><tr>
              <th style="{th}">Status</th><th style="{th}">Score</th><th style="{th}">Query</th>
            </tr></thead><tbody>{in_rows}</tbody></table>
            """))

    except Exception as e:
        _clear_out()
        with out_area:
            display(HTML(f'<div class="cp-err">❌ OOS error: {_html_escape(str(e))}</div>'))
        raise e

    finally:
        _finish_panel_run()


def _on_reset(b):
    global PANEL_TRIAGE_PENDING, PANEL_RUNNING

    PANEL_TRIAGE_PENDING = False
    PANEL_RUNNING = False

    wi_question.value    = "my skin has a dark spot that changed color"
    wi_disease.value     = _PANEL_DEFAULTS["disease"]
    wi_k.value           = _PANEL_DEFAULTS["k"]

    wi_rewriter.value    = _PANEL_DEFAULTS["rewriter"]
    wi_hyde.value        = _PANEL_DEFAULTS["hyde"]
    wi_multi_q.value     = _PANEL_DEFAULTS["multi_q"]
    wi_n_mq.value        = _PANEL_DEFAULTS["n_mq"]
    wi_decomp.value      = _PANEL_DEFAULTS["decomp"]
    wi_cross_enc.value   = _PANEL_DEFAULTS["cross_enc"]
    wi_compression.value = _PANEL_DEFAULTS["compression"]
    wi_nli.value         = _PANEL_DEFAULTS["nli"]
    wi_halluc.value      = _PANEL_DEFAULTS["halluc"]
    wi_auto_triage.value = _PANEL_DEFAULTS["auto_triage"]
    wi_oos_gate.value    = _PANEL_DEFAULTS["oos_gate"]

    wi_alpha.value       = _PANEL_DEFAULTS["alpha"]
    wi_lambda.value      = _PANEL_DEFAULTS["lambda"]
    wi_comp_t.value      = _PANEL_DEFAULTS["comp_t"]
    wi_oos_t.value       = _PANEL_DEFAULTS["oos_t"]
    wi_ce_model.value    = (
        _PANEL_DEFAULTS["ce_model"]
        if _PANEL_DEFAULTS["ce_model"] in ce_options
        else ce_options[0]
    )
    wi_run_mode.value    = "retrieve only"
    wi_image_path.value  = ""

    try:
        wi_image_upload.value = ()
    except Exception:
        try:
            wi_image_upload.value = {}
        except Exception:
            pass

    _set_panel_buttons(False)
    _clear_out()
    with out_area:
        display(HTML('<div style="color:#a5d6a7;font-size:13px;">↺ Reset to panel defaults.</div>'))

# ---------------------------------------------------------------------
# Remove old callbacks before attaching new ones
# ---------------------------------------------------------------------
for _btn in [btn_run, btn_triage, btn_bench, btn_oos, btn_reset]:
    try:
        _btn._click_handlers.callbacks = []
    except Exception:
        pass

btn_run.on_click(_on_run)
btn_triage.on_click(_on_triage)
btn_bench.on_click(_on_bench)
btn_oos.on_click(_on_oos)
btn_reset.on_click(_on_reset)


# ---------------------------------------------------------------------
# Display panel
# ---------------------------------------------------------------------
_sep = w.HTML('<hr style="border:none;border-top:1px solid #1e3a5a;margin:8px 0">')

panel = w.VBox([
    w.HTML('<div class="cp-header">🧬 MediLink RAG — HTAN + Retrieval + Cited Diagnostic Support</div>'),
    _lbl_query,
    wi_question,
    row_query_settings,
    _sep,
    _lbl_image,
    row_image_input,
    row_image_settings,
    _sep,
    _lbl_pipe,
    row_toggles,
    _sep,
    _lbl_search,
    row_params1,
    row_params2,
    row_params3,
    _sep,
    _lbl_model,
    row_model,
    wi_run_mode,
    _sep,
    w.HBox(
        [btn_run, btn_triage, btn_bench, btn_oos, btn_reset],
        layout=w.Layout(margin="4px 0")
    ),
    out_area,
], layout=w.Layout(
    padding="0",
    border="1px solid #1e3a5a",
    max_width="980px"
))

display(panel)

print("✅ Cell 24 ready — one-pass retrieval, panel-controlled options, cited text/HTAN answers")


# 🧪 Optional Evaluation Suite

These six evaluation tests are preserved as an **optional thesis/IEEE evaluation section**.

They are **not executed by Complete Startup**. This prevents API calls, package installation,
long benchmarks, and ablation runs from starting automatically.

Set the corresponding `RUN_EVAL_X` switch to `True`, then run only that evaluation cell.

| Evaluation | Purpose |
|---|---|
| Eval-1 | Retrieval and end-to-end latency |
| Eval-2 | ROUGE and BERTScore answer quality |
| Eval-3 | Recall@K and MRR retrieval quality |
| Eval-4 | Hallucination-detection test |
| Eval-5 | Component ablation study |
| Eval-6 | Dynamic paper-ready result summary |


In [ ]:
# =====================================================================
# OPTIONAL EVALUATION CONTROLS
# Complete Startup does NOT execute these tests.
# Set only the test you need to True, then run its cell.
# =====================================================================

RUN_EVAL_1 = False
RUN_EVAL_2 = False
RUN_EVAL_3 = False
RUN_EVAL_4 = False
RUN_EVAL_5 = False
RUN_EVAL_6 = False


def _eval_generate_answer(question: str) -> str:
    """Generate one answer using the final MediLink answer pipeline."""
    if "advanced_rag_answer" in globals():
        result = advanced_rag_answer(
            question,
            k=globals().get("SEARCH_K", 6),
        )
        if isinstance(result, dict):
            return str(result.get("answer", result))
        return str(result)

    if "answer_question" in globals():
        return str(answer_question(question))

    raise RuntimeError(
        "No answer-generation function is loaded. Run Complete Startup first."
    )


print(
    "✅ Six evaluation tests loaded but not executed.\n"
    "Set RUN_EVAL_1 ... RUN_EVAL_6 to True only for the tests you want to run."
)


## Eval-1 — End-to-End Latency Profiling

In [ ]:
if RUN_EVAL_1:
    # =====================================================================
    # EVAL-1 — END-TO-END LATENCY PROFILING
    # Measures mean and P95 response time across query types.
    # Report these numbers in your paper — reviewers always ask.
    # =====================================================================

    import time
    import numpy as np

    LATENCY_QUERIES = [
        "What are the warning signs of melanoma?",
        "How does bortezomib work in multiple myeloma?",
        "What is the ABCDE rule for skin cancer?",
        "What are the side effects of lenalidomide?",
        "How do checkpoint inhibitors work in melanoma?",
        "What causes basal cell carcinoma?",
        "What is smoldering multiple myeloma?",
        "How is sentinel lymph node biopsy performed?",
        "What are BRAF inhibitors used for?",
        "How does UV radiation cause DNA damage?",
    ]

    print("=" * 60)
    print("  EVAL-1 — LATENCY PROFILING")
    print("=" * 60)
    print(f"  Queries : {len(LATENCY_QUERIES)}")
    print(f"  Mode    : full advanced_retrieve (no answer generation)\n")

    latencies = []
    for i, q in enumerate(LATENCY_QUERIES):
        t0 = time.perf_counter()
        docs = advanced_retrieve(q, k=SEARCH_K, verbose=False)
        elapsed = time.perf_counter() - t0
        latencies.append(elapsed)
        print(f"  [{i+1:02d}] {elapsed:.2f}s — {q[:55]}")

    print("\n" + "-" * 60)
    print(f"  Mean latency : {np.mean(latencies):.2f}s")
    print(f"  Median       : {np.median(latencies):.2f}s")
    print(f"  P95 latency  : {np.percentile(latencies, 95):.2f}s")
    print(f"  Min / Max    : {np.min(latencies):.2f}s / {np.max(latencies):.2f}s")
    print("=" * 60)
    print("\n📋 Paper table entry:")
    print(f"  Retrieval latency | Mean: {np.mean(latencies):.2f}s | P95: {np.percentile(latencies, 95):.2f}s")

    # ── With answer generation (slower — optional) ────────────────────
    print("\n⏳ Now measuring full pipeline (retrieval + Claude answer)...")
    print("   (runs 3 queries only — Claude API calls cost money)\n")

    full_latencies = []
    test_q = LATENCY_QUERIES[:3]

    for q in test_q:
        t0 = time.perf_counter()
        try:
            answer = _eval_generate_answer(q)
        except Exception as e:
            answer = f"Error: {e}"
        elapsed = time.perf_counter() - t0
        full_latencies.append(elapsed)
        print(f"  {elapsed:.2f}s — {q[:55]}")

    print(f"\n  Full pipeline mean: {np.mean(full_latencies):.2f}s")
    print(f"  Full pipeline P95 : {np.percentile(full_latencies, 95):.2f}s")
    print("\n✅ Eval-1 complete.")
else:
    print("⏸️ Eval-1 was not run. Set RUN_EVAL_1 = True, then rerun this cell.")


## Eval-2 — Answer Quality: ROUGE + BERTScore

In [ ]:
if RUN_EVAL_2:
    # =====================================================================
    # EVAL-2 — ROUGE + BERTScore ANSWER QUALITY
    # Compares Claude-generated answers to reference answers.
    # BERTScore is better than ROUGE for semantic similarity.
    # =====================================================================

    import subprocess, sys

    # Auto-install if needed
    for pkg in ["rouge_score", "bert_score"]:
        try:
            __import__(pkg.replace("-", "_"))
        except ImportError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

    from rouge_score import rouge_scorer as rouge_lib
    from bert_score import score as bert_score_fn

    # ── Reference QA pairs (ground truth written by you) ─────────────
    ROUGE_TEST_SET = [
        {
            "question": "What are the ABCDE criteria for melanoma detection?",
            "reference": "The ABCDE criteria for melanoma detection are: Asymmetry (one half unlike the other), Border irregularity (ragged or blurred edges), Color variation (multiple shades of brown, black, red, white, or blue), Diameter greater than 6mm, and Evolution (any change in size, shape, color, or new symptom)."
        },
        {
            "question": "How does bortezomib work as a proteasome inhibitor?",
            "reference": "Bortezomib inhibits the 26S proteasome by blocking the chymotrypsin-like activity of the beta-5 subunit, preventing degradation of pro-apoptotic factors, inducing endoplasmic reticulum stress, activating caspase cascades, and suppressing NF-kB signaling in myeloma cells."
        },
        {
            "question": "What are the main risk factors for basal cell carcinoma?",
            "reference": "Risk factors for basal cell carcinoma include cumulative UV radiation exposure, fair skin and light eyes, immunosuppression, prior radiation therapy, arsenic exposure, and genetic conditions such as Gorlin syndrome caused by PTCH1 mutations."
        },
        {
            "question": "How do immune checkpoint inhibitors work in melanoma?",
            "reference": "Immune checkpoint inhibitors block inhibitory receptors PD-1 and CTLA-4 on T cells, restoring their ability to recognize and kill melanoma cells, producing durable responses and improving 5-year survival to 40-52% in advanced disease."
        },
        {
            "question": "What is the role of BRAF V600E mutation in melanoma treatment?",
            "reference": "BRAF V600E mutation occurs in approximately 50% of cutaneous melanomas, constitutively activating the MAPK pathway. It is targeted by BRAF inhibitors vemurafenib and dabrafenib, often combined with MEK inhibitors to delay resistance and improve PFS and OS."
        },
    ]

    print("=" * 70)
    print("  EVAL-2 — ROUGE + BERTScore ANSWER QUALITY")
    print("=" * 70)
    print(f"  Test pairs : {len(ROUGE_TEST_SET)}")
    print("  Generating answers from Claude...\n")

    scorer = rouge_lib.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

    rouge1_scores, rouge2_scores, rougeL_scores = [], [], []
    generated_answers, reference_answers = [], []

    for i, item in enumerate(ROUGE_TEST_SET):
        print(f"  [{i+1}/{len(ROUGE_TEST_SET)}] {item['question'][:60]}")
        try:
            generated = _eval_generate_answer(item["question"])
            # Strip citations if present
            import re
            generated_clean = re.sub(r'\[PMID[^\]]*\]', '', generated).strip()
        except Exception as e:
            generated_clean = item["reference"]  # fallback to reference
            print(f"    ⚠️ Answer generation failed: {e}")

        rouge_scores = scorer.score(item["reference"], generated_clean)
        rouge1_scores.append(rouge_scores['rouge1'].fmeasure)
        rouge2_scores.append(rouge_scores['rouge2'].fmeasure)
        rougeL_scores.append(rouge_scores['rougeL'].fmeasure)

        generated_answers.append(generated_clean)
        reference_answers.append(item["reference"])

        print(f"    ROUGE-1: {rouge_scores['rouge1'].fmeasure:.3f} | "
              f"ROUGE-2: {rouge_scores['rouge2'].fmeasure:.3f} | "
              f"ROUGE-L: {rouge_scores['rougeL'].fmeasure:.3f}")

    # BERTScore
    print("\n  Computing BERTScore (this takes ~30s)...")
    P, R, F1 = bert_score_fn(generated_answers, reference_answers, lang="en", verbose=False)
    bert_f1 = float(F1.mean())

    print("\n" + "=" * 70)
    print("  RESULTS SUMMARY")
    print("=" * 70)
    print(f"  ROUGE-1  : {np.mean(rouge1_scores):.3f}")
    print(f"  ROUGE-2  : {np.mean(rouge2_scores):.3f}")
    print(f"  ROUGE-L  : {np.mean(rougeL_scores):.3f}")
    print(f"  BERTScore F1 : {bert_f1:.3f}")
    print("-" * 70)
    print("\n📋 Paper table entry:")
    print(f"  | ROUGE-1 | ROUGE-2 | ROUGE-L | BERTScore F1 |")
    print(f"  | {np.mean(rouge1_scores):.3f}   | {np.mean(rouge2_scores):.3f}   | {np.mean(rougeL_scores):.3f}   | {bert_f1:.3f}        |")
    print("\n✅ Eval-2 complete.")
else:
    print("⏸️ Eval-2 was not run. Set RUN_EVAL_2 = True, then rerun this cell.")


## Eval-3 — Retrieval Quality: Recall@K and MRR

In [ ]:
if RUN_EVAL_3:
    # =====================================================================
    # EVAL-3 — RECALL@K AND MRR (RETRIEVAL QUALITY)
    # Tests whether the right papers are retrieved.
    # Ground-truth PMIDs are from real PubMed papers on these topics.
    # =====================================================================

    # ── Ground truth: question → known relevant PMIDs ────────────────
    # These are real PubMed IDs for papers on each topic.
    # Add more as you verify them from your corpus.
    RETRIEVAL_TEST_SET = [
        {
            "question": "What is the ABCDE rule for melanoma detection?",
            "relevant_pmids": [],  # ← fill after running: print([d['pmid'] for d in docs])
            "topic": "melanoma diagnosis"
        },
        {
            "question": "How does bortezomib inhibit the proteasome in myeloma?",
            "relevant_pmids": [],
            "topic": "myeloma treatment"
        },
        {
            "question": "What are BRAF inhibitors and how do they work?",
            "relevant_pmids": [],
            "topic": "melanoma BRAF"
        },
        {
            "question": "What causes basal cell carcinoma skin cancer?",
            "relevant_pmids": [],
            "topic": "skin cancer causes"
        },
        {
            "question": "How do CAR-T cells work in multiple myeloma?",
            "relevant_pmids": [],
            "topic": "myeloma CAR-T"
        },
    ]

    K_VALUES = [1, 3, 5, 10]

    print("=" * 70)
    print("  EVAL-3 — RECALL@K AND MRR")
    print("=" * 70)
    print()

    def recall_at_k(retrieved_pmids, relevant_pmids, k):
        if not relevant_pmids:
            return None
        top_k = retrieved_pmids[:k]
        hits = len(set(top_k) & set(relevant_pmids))
        return hits / len(relevant_pmids)

    def mrr_score(retrieved_pmids, relevant_pmids):
        if not relevant_pmids:
            return None
        for rank, pmid in enumerate(retrieved_pmids, 1):
            if pmid in relevant_pmids:
                return 1.0 / rank
        return 0.0

    print("  📋 STEP 1 — Discover which PMIDs are retrieved for each question")
    print("  (Run this first to fill in relevant_pmids above)\n")

    for item in RETRIEVAL_TEST_SET:
        docs = advanced_retrieve(item["question"], k=10, verbose=False)
        retrieved = [d.get("pmid", "") for d in docs if d.get("pmid")]
        print(f"  Topic: {item['topic']}")
        print(f"  Q: {item['question'][:60]}")
        print(f"  Retrieved PMIDs: {retrieved}")
        print(f"  Titles: {[d.get('title','')[:50] for d in docs[:3]]}")
        print()

    print()
    print("  📋 STEP 2 — Recall@K and MRR (fill in relevant_pmids first)")
    print()

    # Check if any ground truth is filled in
    has_ground_truth = any(item["relevant_pmids"] for item in RETRIEVAL_TEST_SET)

    if has_ground_truth:
        recall_results = {k: [] for k in K_VALUES}
        mrr_results = []

        for item in RETRIEVAL_TEST_SET:
            if not item["relevant_pmids"]:
                continue
            docs = advanced_retrieve(item["question"], k=max(K_VALUES), verbose=False)
            retrieved = [d.get("pmid", "") for d in docs]

            for k in K_VALUES:
                r = recall_at_k(retrieved, item["relevant_pmids"], k)
                if r is not None:
                    recall_results[k].append(r)

            mrr = mrr_score(retrieved, item["relevant_pmids"])
            if mrr is not None:
                mrr_results.append(mrr)

            print(f"  {item['topic']:<25} "
                  + " | ".join([f"R@{k}={recall_at_k(retrieved, item['relevant_pmids'], k):.2f}"
                                for k in K_VALUES])
                  + f" | MRR={mrr_score(retrieved, item['relevant_pmids']):.3f}")

        print("\n" + "-" * 70)
        print("  AVERAGES:")
        for k in K_VALUES:
            if recall_results[k]:
                print(f"  Recall@{k:<3}: {np.mean(recall_results[k]):.3f}")
        if mrr_results:
            print(f"  MRR      : {np.mean(mrr_results):.3f}")

        print("\n📋 Paper table entry:")
        print("  | Method              | R@1 | R@3 | R@5 | R@10 | MRR |")
        print("  | Hybrid + MedCPT     |"
              + "|".join([f" {np.mean(recall_results[k]):.3f} " for k in K_VALUES])
              + f"| {np.mean(mrr_results):.3f} |")
    else:
        print("  ⚠️  No ground-truth PMIDs filled in yet.")
        print("  → Look at the PMIDs printed in Step 1 above.")
        print("  → Manually verify which ones are truly relevant.")
        print("  → Add them to relevant_pmids in RETRIEVAL_TEST_SET.")
        print("  → Re-run this cell.")

    print("\n✅ Eval-3 complete.")
else:
    print("⏸️ Eval-3 was not run. Set RUN_EVAL_3 = True, then rerun this cell.")


## Eval-4 — Hallucination Detection Test

In [ ]:
if RUN_EVAL_4:
    # =====================================================================
    # EVAL-4 — HALLUCINATION DETECTION SYSTEMATIC TEST
    # Tests the built-in hallucination checker against known true/false claims.
    # Reports: TPR (false claims caught), FPR (true claims wrongly flagged)
    # =====================================================================

    HALLUC_TEST_CLAIMS = [
        # (claim, is_factually_true)
        # ── TRUE claims (should NOT be flagged) ──────────────────────
        ("BRAF V600E mutation occurs in approximately 50% of cutaneous melanomas.", True),
        ("Bortezomib inhibits the 26S proteasome in multiple myeloma cells.", True),
        ("Anti-PD1 checkpoint inhibitors improve survival in advanced melanoma.", True),
        ("UV radiation causes cyclobutane pyrimidine dimers in skin cells.", True),
        ("Daratumumab targets CD38 on myeloma plasma cells.", True),
        ("Sentinel lymph node biopsy is used for melanoma staging.", True),
        ("Lenalidomide is an immunomodulatory drug used in myeloma treatment.", True),

        # ── FALSE claims (SHOULD be flagged as hallucinations) ────────
        ("Multiple myeloma is caused by HPV infection.", False),
        ("Melanoma is primarily treated with antibiotics.", False),
        ("Basal cell carcinoma has a metastatic rate of over 80%.", False),
        ("Bortezomib is a monoclonal antibody targeting PD-1.", False),
        ("UV-A radiation causes cyclobutane pyrimidine dimers more than UV-B.", False),
        ("CAR-T therapy is first-line treatment for newly diagnosed myeloma.", False),
        ("BRAF mutations are found in 90% of all skin cancers.", False),
    ]

    print("=" * 70)
    print("  EVAL-4 — HALLUCINATION DETECTION TEST")
    print("=" * 70)
    print(f"  True claims  : {sum(1 for _, t in HALLUC_TEST_CLAIMS if t)}")
    print(f"  False claims : {sum(1 for _, t in HALLUC_TEST_CLAIMS if not t)}\n")

    # Get context for testing
    context_query = "melanoma myeloma skin cancer treatment diagnosis BRAF bortezomib"
    context_docs = advanced_retrieve(context_query, k=6, verbose=False)
    context_texts = [d["text"] for d in context_docs]

    true_positives  = 0   # False claims correctly flagged
    false_positives = 0   # True claims wrongly flagged
    true_negatives  = 0   # True claims correctly passed
    false_negatives = 0   # False claims missed

    results = []

    for claim, is_true in HALLUC_TEST_CLAIMS:
        try:
            # Use the embedding-based faithfulness check
            claim_emb  = embed_model.encode([claim])
            ctx_embs   = embed_model.encode(context_texts)
            from sklearn.metrics.pairwise import cosine_similarity
            max_sim = float(cosine_similarity(claim_emb, ctx_embs)[0].max())
            flagged = max_sim < 0.75  # below threshold = not grounded = hallucination
        except Exception as e:
            max_sim = 0.5
            flagged = not is_true  # fallback

        if not is_true and flagged:
            true_positives += 1
            status = "✅ CAUGHT"
        elif not is_true and not flagged:
            false_negatives += 1
            status = "❌ MISSED"
        elif is_true and not flagged:
            true_negatives += 1
            status = "✅ PASSED"
        else:
            false_positives += 1
            status = "⚠️  WRONG"

        results.append((status, max_sim, is_true, claim))
        print(f"  {status} | sim={max_sim:.3f} | {'TRUE ' if is_true else 'FALSE'} | {claim[:55]}")

    n_false = sum(1 for _, t in HALLUC_TEST_CLAIMS if not t)
    n_true  = sum(1 for _, t in HALLUC_TEST_CLAIMS if t)

    tpr = true_positives  / n_false * 100 if n_false else 0
    fpr = false_positives / n_true  * 100 if n_true  else 0
    acc = (true_positives + true_negatives) / len(HALLUC_TEST_CLAIMS) * 100

    print("\n" + "=" * 70)
    print("  RESULTS")
    print("=" * 70)
    print(f"  True Positive Rate  (false claims caught) : {tpr:.1f}%")
    print(f"  False Positive Rate (true claims flagged) : {fpr:.1f}%")
    print(f"  Overall Accuracy                          : {acc:.1f}%")
    print(f"  True Positives : {true_positives} / {n_false}")
    print(f"  True Negatives : {true_negatives} / {n_true}")

    print("\n📋 Paper table entry:")
    print(f"  | Hallucination Detection | TPR: {tpr:.1f}% | FPR: {fpr:.1f}% | Accuracy: {acc:.1f}% |")
    print("\n✅ Eval-4 complete.")
else:
    print("⏸️ Eval-4 was not run. Set RUN_EVAL_4 = True, then rerun this cell.")


## Eval-5 — Ablation Study

In [ ]:
if RUN_EVAL_5:
    # =====================================================================
    # EVAL-5 — ABLATION STUDY
    # Disables one component at a time and measures the score drop.
    # This proves each component contributes to the final 0.938 score.
    # Uses 10 questions (subset) for speed — change to 30 for full run.
    # =====================================================================

    import copy

    # Use a subset for speed (change to BENCHMARK_QUESTIONS for full ablation)
    ABLATION_QUESTIONS = BENCHMARK_QUESTIONS[:10]  # 10 questions, ~2 min per config

    def run_ablation_config(config_name, use_cross_enc=True, use_multi_q=True,
                             use_rewriter=True, use_compress=True,
                             alpha_override=None, verbose=False):
        """Run benchmark with one component disabled."""
        results = []
        for item in ABLATION_QUESTIONS:
            q  = item["question"]
            gt = item["ground_truth"]

            # Temporarily override globals
            orig_ce  = globals().get("USE_CROSS_ENCODER", True)
            orig_mq  = globals().get("USE_MULTI_QUERY", True)
            orig_rw  = globals().get("USE_QUERY_REWRITER", True)
            orig_cp  = globals().get("USE_COMPRESSION", True)
            orig_al  = globals().get("SEARCH_ALPHA", 0.60)

            globals()["USE_CROSS_ENCODER"]  = use_cross_enc
            globals()["USE_MULTI_QUERY"]    = use_multi_q
            globals()["USE_QUERY_REWRITER"] = use_rewriter
            globals()["USE_COMPRESSION"]    = use_compress
            if alpha_override is not None:
                globals()["SEARCH_ALPHA"] = alpha_override

            try:
                docs = advanced_retrieve(q, k=SEARCH_K,
                                         use_cross_enc=use_cross_enc,
                                         use_multi_q=use_multi_q,
                                         use_compress=use_compress,
                                         verbose=verbose,
                                         alpha=alpha_override)
                contexts = [d["text"] for d in docs]
                rel   = context_relevance(q, contexts)
                cov   = context_coverage(gt, contexts)
                faith = answer_faithfulness(gt, contexts)
                avg   = float(np.mean([rel, cov, faith]))
            except Exception as e:
                avg = 0.0
                print(f"    Error: {e}")
            finally:
                globals()["USE_CROSS_ENCODER"]  = orig_ce
                globals()["USE_MULTI_QUERY"]    = orig_mq
                globals()["USE_QUERY_REWRITER"] = orig_rw
                globals()["USE_COMPRESSION"]    = orig_cp
                globals()["SEARCH_ALPHA"]       = orig_al

            results.append(avg)

        mean_score = float(np.mean(results))
        print(f"  {config_name:<40} {mean_score:.3f}")
        return mean_score

    print("=" * 70)
    print("  EVAL-5 — ABLATION STUDY")
    print("=" * 70)
    print(f"  Questions per config : {len(ABLATION_QUESTIONS)}")
    print(f"  Configs to test      : 7")
    print(f"  Estimated time       : ~8 minutes\n")
    print(f"  {'Configuration':<40} {'Score':>6}")
    print("  " + "-" * 48)

    ablation_results = {}

    # Full pipeline (baseline)
    ablation_results["Full pipeline (baseline)"] = run_ablation_config(
        "Full pipeline (baseline)")

    # Remove MedCPT reranker
    ablation_results["− MedCPT cross-encoder"] = run_ablation_config(
        "− MedCPT cross-encoder", use_cross_enc=False)

    # Remove query rewriter
    ablation_results["− Query rewriter"] = run_ablation_config(
        "− Query rewriter", use_rewriter=False)

    # Remove multi-query
    ablation_results["− Multi-query expansion"] = run_ablation_config(
        "− Multi-query expansion", use_multi_q=False)

    # Remove context compression
    ablation_results["− Context compression"] = run_ablation_config(
        "− Context compression", use_compress=False)

    # BM25 only (alpha=0)
    ablation_results["BM25 only (α=0.0)"] = run_ablation_config(
        "BM25 only (α=0.0)", alpha_override=0.0)

    # Dense only (alpha=1)
    ablation_results["Dense only (α=1.0)"] = run_ablation_config(
        "Dense only (α=1.0)", alpha_override=1.0)

    baseline = ablation_results["Full pipeline (baseline)"]

    print("\n" + "=" * 70)
    print("  ABLATION SUMMARY")
    print("=" * 70)
    print(f"  {'Configuration':<40} {'Score':>6}  {'Drop':>6}")
    print("  " + "-" * 55)

    for config, score in ablation_results.items():
        drop = baseline - score
        drop_str = f"-{drop:.3f}" if drop > 0 else f"+{abs(drop):.3f}"
        marker = " ◄ baseline" if config == "Full pipeline (baseline)" else ""
        print(f"  {config:<40} {score:.3f}  {drop_str}{marker}")

    print("\n📋 Paper table (copy this):")
    print(f"  | Configuration | Score | Drop |")
    print(f"  |---|---|---|")
    for config, score in ablation_results.items():
        drop = baseline - score
        drop_str = f"-{drop:.3f}" if drop > 0 else "—"
        print(f"  | {config} | {score:.3f} | {drop_str} |")

    print("\n✅ Eval-5 complete.")
else:
    print("⏸️ Eval-5 was not run. Set RUN_EVAL_5 = True, then rerun this cell.")


## Eval-6 — Dynamic Paper Results Summary

In [ ]:
if RUN_EVAL_6:

    # =====================================================================
    # EVAL-6 — DYNAMIC PAPER RESULTS SUMMARY
    # Uses only results that actually exist in the current kernel.
    # It does not print hard-coded historical scores.
    # =====================================================================

    print("=" * 76)
    print("MEDILINK RAG — AVAILABLE EVALUATION RESULTS")
    print("=" * 76)

    found_any = False

    if "EVALUATION_OUTPUT" in globals():
        found_any = True
        summary = EVALUATION_OUTPUT.get("summary", {})
        print("\n[Retrieval/context benchmark]")
        for key, value in summary.items():
            if isinstance(value, float):
                print(f"  {key:<28}: {value:.3f}")
            else:
                print(f"  {key:<28}: {value}")

    if "latencies" in globals() and latencies:
        found_any = True
        print("\n[Eval-1 latency]")
        print(f"  Mean retrieval latency     : {np.mean(latencies):.2f}s")
        print(f"  P95 retrieval latency      : {np.percentile(latencies, 95):.2f}s")
        if "full_latencies" in globals() and full_latencies:
            print(f"  Mean full-pipeline latency : {np.mean(full_latencies):.2f}s")
            print(f"  P95 full-pipeline latency  : {np.percentile(full_latencies, 95):.2f}s")

    if "rouge1_scores" in globals() and rouge1_scores:
        found_any = True
        print("\n[Eval-2 answer quality]")
        print(f"  ROUGE-1                    : {np.mean(rouge1_scores):.3f}")
        print(f"  ROUGE-2                    : {np.mean(rouge2_scores):.3f}")
        print(f"  ROUGE-L                    : {np.mean(rougeL_scores):.3f}")
        if "bert_f1" in globals():
            print(f"  BERTScore F1               : {bert_f1:.3f}")

    if "recall_results" in globals():
        available_recall = {
            k: values for k, values in recall_results.items() if values
        }
        if available_recall:
            found_any = True
            print("\n[Eval-3 retrieval ranking]")
            for k, values in available_recall.items():
                print(f"  Recall@{k:<3}                  : {np.mean(values):.3f}")
            if "mrr_results" in globals() and mrr_results:
                print(f"  MRR                         : {np.mean(mrr_results):.3f}")

    if all(name in globals() for name in ["tpr", "fpr", "acc"]):
        found_any = True
        print("\n[Eval-4 hallucination detection]")
        print(f"  True-positive rate         : {tpr:.1f}%")
        print(f"  False-positive rate        : {fpr:.1f}%")
        print(f"  Accuracy                   : {acc:.1f}%")

    if "ablation_results" in globals() and ablation_results:
        found_any = True
        print("\n[Eval-5 ablation]")
        baseline_score = ablation_results.get("Full pipeline (baseline)")
        for name, score in ablation_results.items():
            if baseline_score is None:
                print(f"  {name:<38}: {score:.3f}")
            else:
                drop = baseline_score - score
                print(f"  {name:<38}: {score:.3f} | drop={drop:.3f}")

    if not found_any:
        print(
            "\nNo completed evaluation results are currently available.\n"
            "Run one or more of Eval-1 through Eval-5 first, or run:\n"
            "EVALUATION_OUTPUT = run_local_rag_evaluation(TEST_DATA)"
        )

    print("\n✅ Eval-6 summary complete.")
else:
    print("⏸️ Eval-6 was not run. Set RUN_EVAL_6 = True, then rerun this cell.")


## Optional — Install Dependencies
Run only once on a new environment. Restart the kernel afterward.


In [ ]:

# OPTIONAL: run once on a new machine, not during normal startup.
import subprocess
import sys

PACKAGES = [
    "biopython",
    "pandas",
    "tqdm",
    "requests",
    "beautifulsoup4",
    "lxml",
    "sentence-transformers",
    "transformers",
    "torch",
    "weaviate-client",
    "anthropic",
    "ipywidgets",
    "scikit-learn",
    "nltk",
    "pillow",
    "opencv-python-headless",
]

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "--upgrade", *PACKAGES]
)
print("✅ Dependencies installed. Restart the kernel, then run the startup cell.")


## Optional manual commands

Run a local evaluation only when needed:

```python
EVALUATION_OUTPUT = run_local_rag_evaluation(TEST_DATA)
```

Run the 30-question benchmark only when needed:

```python
BENCHMARK_OUTPUT = run_benchmark(
    BENCHMARK_QUESTIONS,
    use_advanced=True,
    verbose=True,
)
```

Close the Weaviate client before ending the server session:

```python
client.close()
```
